### Direction Model Research

This notebook compares candidate models for predicting the sign of the next-session overnight return.

Target:

- `actual_direction`
- `-1` for negative overnight return
- `+1` for zero or positive overnight return

Models will be compared using the same training and validation data.

The test set will not be used until the final direction model is frozen.

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

In [ ]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.6f}")

In [ ]:
NOTEBOOK_DIR = Path.cwd().resolve()

if NOTEBOOK_DIR.name == "notebooks":
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

PROCESSED_DATA_DIR = PROJECT_ROOT / "outputs" / "processed_data"
EXPERIMENTS_DIR = PROJECT_ROOT / "outputs" / "experiments"
MODELS_DIR = PROJECT_ROOT / "outputs" / "models"
CONFIGS_DIR = PROJECT_ROOT / "outputs" / "configs"

EXPERIMENTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
CONFIGS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PANEL_PATH = PROCESSED_DATA_DIR / "model_features_with_splits_v1.parquet"

print("Project root:", PROJECT_ROOT)
print("Model panel:", MODEL_PANEL_PATH)

In [ ]:
assert MODEL_PANEL_PATH.exists(), f"Model panel not found: {MODEL_PANEL_PATH}"

print("Saved modelling panel found.")

In [ ]:
model_df = pd.read_parquet(MODEL_PANEL_PATH)

model_df = model_df.sort_values(["pred_date", "symbol"]).reset_index(drop=True)

print("Shape:", model_df.shape)
print("Date range:", model_df["pred_date"].min(), "to", model_df["pred_date"].max())
print("Symbols:", model_df["symbol"].nunique())

model_df.head()

In [ ]:
split_summary = model_df.groupby("split").agg(
    rows=("pred_date", "size"),
    dates=("pred_date", "nunique"),
    symbols=("symbol", "nunique"),
    start_date=("pred_date", "min"),
    end_date=("pred_date", "max"),
)

split_summary

In [ ]:
daily_feature_columns = [
    "lagged_overnight_return_1d",
    "return_1d",
    "return_5d",
    "return_20d",
    "daily_volatility_20d",
    "overnight_std_20d",
    "daily_volatility_5d",
    "gap_mean_20d",
    "gap_positive_fraction_20d",
    "volume_zscore_20d",
    "volume_trend_5d_20d",
    "calendar_gap_days",
    "day_of_week",
]

minute_feature_columns = [
    "intraday_realized_volatility_pct",
    "close_auction_return_concentration",
    "close_auction_volume_concentration",
    "morning_vs_afternoon_return_pct",
    "close_vwap_deviation_pct",
    "amihud_illiquidity",
]

cross_sectional_feature_columns = [
    "return_1d_rank_pct",
    "return_1d_zscore",
    "rolling_beta_to_universe_60d",
    "cross_sectional_return_dispersion",
    "market_breadth",
    "aggregate_universe_volatility",
]

base_numeric_feature_columns = daily_feature_columns + minute_feature_columns + cross_sectional_feature_columns

print("Daily features:", len(daily_feature_columns))
print("Minute features:", len(minute_feature_columns))
print("Cross-sectional features:", len(cross_sectional_feature_columns))
print("Total numeric features:", len(base_numeric_feature_columns))

In [ ]:
missing_feature_columns = sorted(set(base_numeric_feature_columns) - set(model_df.columns))

assert not missing_feature_columns, f"Missing feature columns: {missing_feature_columns}"

print("All expected Version 1 features are available.")

In [ ]:
metadata_columns = [
    "symbol",
    "pred_date",
    "target_date",
    "split",
]

target_column = "actual_direction"

leakage_columns = [
    "next_open",
    "actual_return_pct",
    "actual_direction",
    "actual_magnitude_pct",
    "target_date",
    "universe_mean_pct",
]

In [ ]:
direction_df = model_df.loc[
    model_df["split"].isin(["train", "valid", "test"])
].copy()

initial_rows = len(direction_df)

direction_df = direction_df.dropna(subset=daily_feature_columns).reset_index(drop=True)

print("Initial rows:", initial_rows)
print("Rows after removing daily warm-up periods:", len(direction_df))
print("Rows removed:", initial_rows - len(direction_df))

In [ ]:
direction_missingness = direction_df[base_numeric_feature_columns].isna().sum().to_frame("missing_count")

direction_missingness["missing_pct"] = direction_missingness["missing_count"] / len(direction_df) * 100

direction_missingness.sort_values("missing_pct", ascending=False)

In [ ]:
train_df = direction_df.loc[direction_df["split"] == "train"].copy()
valid_df = direction_df.loc[direction_df["split"] == "valid"].copy()
test_df = direction_df.loc[direction_df["split"] == "test"].copy()

print("Train rows:", len(train_df))
print("Validation rows:", len(valid_df))
print("Test rows:", len(test_df))

In [ ]:
train_metadata = train_df[metadata_columns].reset_index(drop=True)
valid_metadata = valid_df[metadata_columns].reset_index(drop=True)
test_metadata = test_df[metadata_columns].reset_index(drop=True)

In [ ]:
y_train = train_df[target_column].astype("int8").reset_index(drop=True)
y_valid = valid_df[target_column].astype("int8").reset_index(drop=True)
y_test = test_df[target_column].astype("int8").reset_index(drop=True)

y_train_binary = (y_train == 1).astype("int8")
y_valid_binary = (y_valid == 1).astype("int8")
y_test_binary = (y_test == 1).astype("int8")

In [ ]:
X_train_numeric = train_df[base_numeric_feature_columns].reset_index(drop=True)
X_valid_numeric = valid_df[base_numeric_feature_columns].reset_index(drop=True)
X_test_numeric = test_df[base_numeric_feature_columns].reset_index(drop=True)

print("X train:", X_train_numeric.shape)
print("X valid:", X_valid_numeric.shape)
print("X test:", X_test_numeric.shape)

In [ ]:
lightgbm_feature_columns = base_numeric_feature_columns + ["symbol"]

X_train_lgb = train_df[lightgbm_feature_columns].copy().reset_index(drop=True)
X_valid_lgb = valid_df[lightgbm_feature_columns].copy().reset_index(drop=True)
X_test_lgb = test_df[lightgbm_feature_columns].copy().reset_index(drop=True)

all_symbols = sorted(direction_df["symbol"].astype(str).unique())

symbol_dtype = pd.CategoricalDtype(categories=all_symbols)

X_train_lgb["symbol"] = X_train_lgb["symbol"].astype(str).astype(symbol_dtype)
X_valid_lgb["symbol"] = X_valid_lgb["symbol"].astype(str).astype(symbol_dtype)
X_test_lgb["symbol"] = X_test_lgb["symbol"].astype(str).astype(symbol_dtype)

print(X_train_lgb["symbol"].dtype)

In [ ]:
class_distribution = pd.DataFrame(
    {
        "train": y_train.value_counts(normalize=True).sort_index(),
        "valid": y_valid.value_counts(normalize=True).sort_index(),
        "test": y_test.value_counts(normalize=True).sort_index(),
    }
)

class_distribution

In [ ]:
assert len(X_train_numeric) == len(y_train)
assert len(X_valid_numeric) == len(y_valid)
assert len(X_test_numeric) == len(y_test)

assert y_train.isin([-1, 1]).all()
assert y_valid.isin([-1, 1]).all()
assert y_test.isin([-1, 1]).all()

assert set(train_metadata["pred_date"]).isdisjoint(set(valid_metadata["pred_date"]))
assert set(valid_metadata["pred_date"]).isdisjoint(set(test_metadata["pred_date"]))

print("Direction modelling data preparation complete.")

### Direction Model Benchmarking

All candidate models are trained on the same training period and evaluated on the same validation period.

Primary selection metric:

- `direction_score`

Supporting metrics:

- directional return
- hit rate
- precision
- recall
- F1
- ROC-AUC
- prediction balance

In [ ]:
import time

import lightgbm as lgb

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
direction_experiment_results = []
direction_prediction_store = {}

In [ ]:
def evaluate_direction_model(model_name, actual_return_pct, actual_direction, predicted_direction, probability_up, runtime_seconds):
    actual_return_pct = np.asarray(actual_return_pct, dtype=float)
    actual_direction = np.asarray(actual_direction, dtype=int)
    predicted_direction = np.asarray(predicted_direction, dtype=int)
    probability_up = np.asarray(probability_up, dtype=float)

    correct = (actual_direction == predicted_direction).astype(int)

    absolute_return_sum = np.abs(actual_return_pct).sum()

    if absolute_return_sum > 0:
        direction_score = np.sum(predicted_direction * actual_return_pct) / absolute_return_sum
    else:
        direction_score = np.nan

    directional_return_pct = np.mean(predicted_direction * actual_return_pct)

    hit_rate = np.mean(correct)

    precision_up = precision_score(
        actual_direction,
        predicted_direction,
        pos_label=1,
        zero_division=0,
    )

    recall_up = recall_score(
        actual_direction,
        predicted_direction,
        pos_label=1,
        zero_division=0,
    )

    f1_up = f1_score(
        actual_direction,
        predicted_direction,
        pos_label=1,
        zero_division=0,
    )

    accuracy = accuracy_score(
        actual_direction,
        predicted_direction,
    )

    if len(np.unique(actual_direction)) == 2:
        roc_auc = roc_auc_score(
            (actual_direction == 1).astype(int),
            probability_up,
        )
    else:
        roc_auc = np.nan

    predicted_up_fraction = np.mean(predicted_direction == 1)
    predicted_down_fraction = np.mean(predicted_direction == -1)

    result = {
        "model": model_name,
        "direction_score": direction_score,
        "directional_return_pct": directional_return_pct,
        "hit_rate": hit_rate,
        "accuracy": accuracy,
        "precision_up": precision_up,
        "recall_up": recall_up,
        "f1_up": f1_up,
        "roc_auc": roc_auc,
        "predicted_up_fraction": predicted_up_fraction,
        "predicted_down_fraction": predicted_down_fraction,
        "runtime_seconds": runtime_seconds,
    }

    return result

In [ ]:
valid_actual_return_pct = valid_df["actual_return_pct"].reset_index(drop=True)

assert len(valid_actual_return_pct) == len(y_valid)

print("Validation observations:", len(valid_actual_return_pct))

#### Baseline 1 — Always Predict Up

This benchmark predicts `+1` for every validation observation.

It measures the performance available from the natural upward class imbalance alone.

In [ ]:
always_up_start_time = time.perf_counter()

always_up_prediction = np.ones(len(valid_df), dtype=int)

train_up_rate = np.mean(y_train == 1)

always_up_probability = np.full(
    len(valid_df),
    train_up_rate,
    dtype=float,
)

always_up_runtime = time.perf_counter() - always_up_start_time

In [ ]:
always_up_result = evaluate_direction_model(
    model_name="Always Up",
    actual_return_pct=valid_actual_return_pct,
    actual_direction=y_valid,
    predicted_direction=always_up_prediction,
    probability_up=always_up_probability,
    runtime_seconds=always_up_runtime,
)

direction_experiment_results.append(always_up_result)

direction_prediction_store["Always Up"] = {
    "predicted_direction": always_up_prediction,
    "probability_up": always_up_probability,
}

pd.DataFrame([always_up_result]).T

In [ ]:
confusion_matrix(
    y_valid,
    always_up_prediction,
    labels=[-1, 1],
)

### Baseline 2 — Logistic Regression

Logistic regression provides a simple and interpretable linear benchmark.

Preprocessing is fitted only on training data:

- median imputation for numeric features
- standardisation for numeric features
- one-hot encoding for symbol

In [ ]:
logistic_feature_columns = base_numeric_feature_columns + ["symbol"]

X_train_logistic = train_df[logistic_feature_columns].copy().reset_index(drop=True)
X_valid_logistic = valid_df[logistic_feature_columns].copy().reset_index(drop=True)

numeric_columns_logistic = base_numeric_feature_columns
categorical_columns_logistic = ["symbol"]

In [ ]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("one_hot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

logistic_preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_columns_logistic),
        ("categorical", categorical_pipeline, categorical_columns_logistic),
    ]
)

In [ ]:
logistic_model = Pipeline(
    steps=[
        ("preprocessor", logistic_preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42,
            ),
        ),
    ]
)

In [ ]:
logistic_start_time = time.perf_counter()

logistic_model.fit(
    X_train_logistic,
    y_train_binary,
)

logistic_runtime = time.perf_counter() - logistic_start_time

print(f"Training time: {logistic_runtime:.2f} seconds")

In [ ]:
logistic_valid_probability = logistic_model.predict_proba(
    X_valid_logistic
)[:, 1]

logistic_valid_prediction = np.where(
    logistic_valid_probability >= 0.5,
    1,
    -1,
)

In [ ]:
logistic_result = evaluate_direction_model(
    model_name="Logistic Regression",
    actual_return_pct=valid_actual_return_pct,
    actual_direction=y_valid,
    predicted_direction=logistic_valid_prediction,
    probability_up=logistic_valid_probability,
    runtime_seconds=logistic_runtime,
)

direction_experiment_results.append(logistic_result)

direction_prediction_store["Logistic Regression"] = {
    "predicted_direction": logistic_valid_prediction,
    "probability_up": logistic_valid_probability,
}

pd.DataFrame([logistic_result]).T

In [ ]:
confusion_matrix(
    y_valid,
    logistic_valid_prediction,
    labels=[-1, 1],
)

### Baseline 3 — LightGBM

LightGBM is the primary nonlinear candidate.

It can learn:

- thresholds
- nonlinear relationships
- interactions between features
- stock-specific effects through the categorical symbol feature

In [ ]:
lightgbm_params = {
    "objective": "binary",
    "metric": "binary_logloss",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "feature_fraction": 0.80,
    "bagging_fraction": 0.80,
    "bagging_freq": 5,
    "seed": 42,
    "feature_fraction_seed": 42,
    "bagging_seed": 42,
    "data_random_seed": 42,
    "verbosity": -1,
}

In [ ]:
lightgbm_train_dataset = lgb.Dataset(
    X_train_lgb,
    label=y_train_binary,
    categorical_feature=["symbol"],
    free_raw_data=False,
)

lightgbm_valid_dataset = lgb.Dataset(
    X_valid_lgb,
    label=y_valid_binary,
    categorical_feature=["symbol"],
    free_raw_data=False,
)

In [ ]:
lightgbm_start_time = time.perf_counter()

lightgbm_model = lgb.train(
    params=lightgbm_params,
    train_set=lightgbm_train_dataset,
    valid_sets=[
        lightgbm_train_dataset,
        lightgbm_valid_dataset,
    ],
    valid_names=[
        "train",
        "valid",
    ],
    num_boost_round=500,
    callbacks=[
        lgb.early_stopping(50),
        lgb.log_evaluation(50),
    ],
)

lightgbm_runtime = time.perf_counter() - lightgbm_start_time

print(f"Training time: {lightgbm_runtime:.2f} seconds")
print("Best iteration:", lightgbm_model.best_iteration)

In [ ]:
lightgbm_valid_probability = lightgbm_model.predict(
    X_valid_lgb,
    num_iteration=lightgbm_model.best_iteration,
)

lightgbm_valid_prediction = np.where(
    lightgbm_valid_probability >= 0.5,
    1,
    -1,
)

In [ ]:
lightgbm_result = evaluate_direction_model(
    model_name="LightGBM",
    actual_return_pct=valid_actual_return_pct,
    actual_direction=y_valid,
    predicted_direction=lightgbm_valid_prediction,
    probability_up=lightgbm_valid_probability,
    runtime_seconds=lightgbm_runtime,
)

direction_experiment_results.append(lightgbm_result)

direction_prediction_store["LightGBM"] = {
    "predicted_direction": lightgbm_valid_prediction,
    "probability_up": lightgbm_valid_probability,
}

pd.DataFrame([lightgbm_result]).T

In [ ]:
confusion_matrix(
    y_valid,
    lightgbm_valid_prediction,
    labels=[-1, 1],
)

In [ ]:
direction_comparison_df = pd.DataFrame(direction_experiment_results)

direction_comparison_df = direction_comparison_df.sort_values(
    "direction_score",
    ascending=False,
).reset_index(drop=True)

direction_comparison_df

In [ ]:
direction_comparison_df[
    [
        "model",
        "direction_score",
        "directional_return_pct",
        "hit_rate",
        "precision_up",
        "recall_up",
        "f1_up",
        "roc_auc",
        "predicted_up_fraction",
        "predicted_down_fraction",
        "runtime_seconds",
    ]
]

#### LightGBM Improvement Experiments

Baseline LightGBM predicts an up move whenever:

`probability_up >= 0.50`

It currently predicts up for approximately 98% of validation rows.

We will test:

1. Alternative decision thresholds
2. Class-balanced LightGBM
3. Alternative thresholds for the balanced model

In [ ]:
def evaluate_thresholds(model_name, probability_up, actual_return_pct, actual_direction, thresholds):
    threshold_results = []

    for threshold in thresholds:
        predicted_direction = np.where(probability_up >= threshold, 1, -1)

        result = evaluate_direction_model(
            model_name=model_name,
            actual_return_pct=actual_return_pct,
            actual_direction=actual_direction,
            predicted_direction=predicted_direction,
            probability_up=probability_up,
            runtime_seconds=0.0,
        )

        result["threshold"] = threshold
        threshold_results.append(result)

    return pd.DataFrame(threshold_results)

In [ ]:
threshold_grid = np.round(
    np.arange(0.40, 0.81, 0.01),
    2,
)

print("Thresholds tested:", len(threshold_grid))
print("Range:", threshold_grid.min(), "to", threshold_grid.max())

In [ ]:
lightgbm_threshold_results = evaluate_thresholds(
    model_name="LightGBM Threshold Search",
    probability_up=lightgbm_valid_probability,
    actual_return_pct=valid_actual_return_pct,
    actual_direction=y_valid,
    thresholds=threshold_grid,
)

lightgbm_threshold_results = lightgbm_threshold_results.sort_values(
    "direction_score",
    ascending=False,
).reset_index(drop=True)

lightgbm_threshold_results.head(5)

In [ ]:
best_lightgbm_threshold_row = lightgbm_threshold_results.iloc[0]

best_lightgbm_threshold = float(
    best_lightgbm_threshold_row["threshold"]
)

print("Best threshold:", best_lightgbm_threshold)
print("Direction score:", best_lightgbm_threshold_row["direction_score"])
print("Directional return:", best_lightgbm_threshold_row["directional_return_pct"])
print("Hit rate:", best_lightgbm_threshold_row["hit_rate"])
print("Predicted up fraction:", best_lightgbm_threshold_row["predicted_up_fraction"])
print("Predicted down fraction:", best_lightgbm_threshold_row["predicted_down_fraction"])

In [ ]:
lightgbm_default_result = direction_comparison_df.loc[
    direction_comparison_df["model"] == "LightGBM"
].copy()

lightgbm_optimized_result = best_lightgbm_threshold_row.to_frame().T

lightgbm_threshold_comparison = pd.concat(
    [
        lightgbm_default_result.assign(threshold=0.50),
        lightgbm_optimized_result,
    ],
    ignore_index=True,
)

lightgbm_threshold_comparison[
    [
        "model",
        "threshold",
        "direction_score",
        "directional_return_pct",
        "hit_rate",
        "precision_up",
        "recall_up",
        "f1_up",
        "predicted_up_fraction",
        "predicted_down_fraction",
    ]
]

In [ ]:
lightgbm_optimized_prediction = np.where(
    lightgbm_valid_probability >= best_lightgbm_threshold,
    1,
    -1,
)

direction_prediction_store["LightGBM Optimized Threshold"] = {
    "predicted_direction": lightgbm_optimized_prediction,
    "probability_up": lightgbm_valid_probability,
    "threshold": best_lightgbm_threshold,
}

#### Class-Balanced LightGBM

The training set contains substantially more positive than negative overnight returns.

This experiment gives additional weight to the minority negative class during training.

In [ ]:
positive_count = int((y_train_binary == 1).sum())
negative_count = int((y_train_binary == 0).sum())

scale_pos_weight = negative_count / positive_count

print("Positive observations:", positive_count)
print("Negative observations:", negative_count)
print("Positive-class weight:", scale_pos_weight)

In [ ]:
balanced_lightgbm_params = lightgbm_params.copy()

balanced_lightgbm_params["scale_pos_weight"] = scale_pos_weight

In [ ]:
balanced_lightgbm_start_time = time.perf_counter()

balanced_lightgbm_model = lgb.train(
    params=balanced_lightgbm_params,
    train_set=lightgbm_train_dataset,
    valid_sets=[
        lightgbm_train_dataset,
        lightgbm_valid_dataset,
    ],
    valid_names=[
        "train",
        "valid",
    ],
    num_boost_round=500,
    callbacks=[
        lgb.early_stopping(50),
        lgb.log_evaluation(50),
    ],
)

balanced_lightgbm_runtime = (
    time.perf_counter() - balanced_lightgbm_start_time
)

print(f"Training time: {balanced_lightgbm_runtime:.2f} seconds")
print("Best iteration:", balanced_lightgbm_model.best_iteration)

In [ ]:
balanced_lightgbm_valid_probability = balanced_lightgbm_model.predict(
    X_valid_lgb,
    num_iteration=balanced_lightgbm_model.best_iteration,
)

balanced_lightgbm_default_prediction = np.where(
    balanced_lightgbm_valid_probability >= 0.50,
    1,
    -1,
)

In [ ]:
balanced_lightgbm_default_result = evaluate_direction_model(
    model_name="Balanced LightGBM",
    actual_return_pct=valid_actual_return_pct,
    actual_direction=y_valid,
    predicted_direction=balanced_lightgbm_default_prediction,
    probability_up=balanced_lightgbm_valid_probability,
    runtime_seconds=balanced_lightgbm_runtime,
)

pd.DataFrame([balanced_lightgbm_default_result]).T

In [ ]:
confusion_matrix(
    y_valid,
    balanced_lightgbm_default_prediction,
    labels=[-1, 1],
)

In [ ]:
balanced_threshold_results = evaluate_thresholds(
    model_name="Balanced LightGBM Threshold Search",
    probability_up=balanced_lightgbm_valid_probability,
    actual_return_pct=valid_actual_return_pct,
    actual_direction=y_valid,
    thresholds=threshold_grid,
)

balanced_threshold_results = balanced_threshold_results.sort_values(
    "direction_score",
    ascending=False,
).reset_index(drop=True)

balanced_threshold_results.head(5)

In [ ]:
best_balanced_threshold_row = balanced_threshold_results.iloc[0]

best_balanced_threshold = float(
    best_balanced_threshold_row["threshold"]
)

print("Best balanced threshold:", best_balanced_threshold)
print("Direction score:", best_balanced_threshold_row["direction_score"])
print("Directional return:", best_balanced_threshold_row["directional_return_pct"])
print("Hit rate:", best_balanced_threshold_row["hit_rate"])
print("Predicted up fraction:", best_balanced_threshold_row["predicted_up_fraction"])
print("Predicted down fraction:", best_balanced_threshold_row["predicted_down_fraction"])

In [ ]:
balanced_lightgbm_optimized_prediction = np.where(
    balanced_lightgbm_valid_probability >= best_balanced_threshold,
    1,
    -1,
)

direction_prediction_store["Balanced LightGBM Optimized Threshold"] = {
    "predicted_direction": balanced_lightgbm_optimized_prediction,
    "probability_up": balanced_lightgbm_valid_probability,
    "threshold": best_balanced_threshold,
}

In [ ]:
lightgbm_improvement_results = pd.DataFrame(
    [
        {
            **direction_comparison_df.loc[
                direction_comparison_df["model"] == "LightGBM"
            ].iloc[0].to_dict(),
            "experiment": "Baseline LightGBM",
            "threshold": 0.50,
        },
        {
            **best_lightgbm_threshold_row.to_dict(),
            "experiment": "Baseline LightGBM + Optimized Threshold",
        },
        {
            **balanced_lightgbm_default_result,
            "experiment": "Balanced LightGBM",
            "threshold": 0.50,
        },
        {
            **best_balanced_threshold_row.to_dict(),
            "experiment": "Balanced LightGBM + Optimized Threshold",
        },
    ]
)

lightgbm_improvement_results = lightgbm_improvement_results.sort_values(
    "direction_score",
    ascending=False,
).reset_index(drop=True)

lightgbm_improvement_results[
    [
        "experiment",
        "threshold",
        "direction_score",
        "directional_return_pct",
        "hit_rate",
        "precision_up",
        "recall_up",
        "f1_up",
        "roc_auc",
        "predicted_up_fraction",
        "predicted_down_fraction",
    ]
]

In [ ]:
lightgbm_improvement_results_path = (
    EXPERIMENTS_DIR
    / "direction_lightgbm_improvements_v1.csv"
)

lightgbm_improvement_results.to_csv(
    lightgbm_improvement_results_path,
    index=False,
)

print("Saved:", lightgbm_improvement_results_path)

In [ ]:
lightgbm_threshold_results.to_csv(
    EXPERIMENTS_DIR
    / "direction_lightgbm_threshold_search_v1.csv",
    index=False,
)

balanced_threshold_results.to_csv(
    EXPERIMENTS_DIR
    / "direction_balanced_lightgbm_threshold_search_v1.csv",
    index=False,
)

print("Threshold-search results saved.")

In [ ]:
# print("=" * 100)
# print("DIRECTION MODEL UPDATE SUMMARY")
# print("=" * 100)

# print("\n1. LIGHTGBM EXPERIMENT COMPARISON")
# print("-" * 100)

# comparison_columns = [
#     "experiment",
#     "threshold",
#     "direction_score",
#     "directional_return_pct",
#     "hit_rate",
#     "precision_up",
#     "recall_up",
#     "f1_up",
#     "roc_auc",
#     "predicted_up_fraction",
#     "predicted_down_fraction",
# ]

# print(
#     lightgbm_improvement_results[comparison_columns]
#     .to_string(index=False)
# )

# print("\n2. BEST BASELINE LIGHTGBM THRESHOLD")
# print("-" * 100)

# print(f"Threshold                 : {best_lightgbm_threshold:.4f}")
# print(f"Direction score           : {best_lightgbm_threshold_row['direction_score']:.6f}")
# print(f"Directional return pct    : {best_lightgbm_threshold_row['directional_return_pct']:.6f}")
# print(f"Hit rate                  : {best_lightgbm_threshold_row['hit_rate']:.6f}")
# print(f"Predicted up fraction     : {best_lightgbm_threshold_row['predicted_up_fraction']:.6f}")
# print(f"Predicted down fraction   : {best_lightgbm_threshold_row['predicted_down_fraction']:.6f}")

# print("\n3. BEST BALANCED LIGHTGBM THRESHOLD")
# print("-" * 100)

# print(f"Threshold                 : {best_balanced_threshold:.4f}")
# print(f"Direction score           : {best_balanced_threshold_row['direction_score']:.6f}")
# print(f"Directional return pct    : {best_balanced_threshold_row['directional_return_pct']:.6f}")
# print(f"Hit rate                  : {best_balanced_threshold_row['hit_rate']:.6f}")
# print(f"Predicted up fraction     : {best_balanced_threshold_row['predicted_up_fraction']:.6f}")
# print(f"Predicted down fraction   : {best_balanced_threshold_row['predicted_down_fraction']:.6f}")

# print("\n4. CONFUSION MATRICES")
# print("-" * 100)

# baseline_cm = confusion_matrix(
#     y_valid,
#     lightgbm_valid_prediction,
#     labels=[-1, 1],
# )

# baseline_optimized_cm = confusion_matrix(
#     y_valid,
#     lightgbm_optimized_prediction,
#     labels=[-1, 1],
# )

# balanced_default_cm = confusion_matrix(
#     y_valid,
#     balanced_lightgbm_default_prediction,
#     labels=[-1, 1],
# )

# balanced_optimized_cm = confusion_matrix(
#     y_valid,
#     balanced_lightgbm_optimized_prediction,
#     labels=[-1, 1],
# )

# print("\nBaseline LightGBM, threshold 0.50")
# print(baseline_cm)

# print("\nBaseline LightGBM, optimized threshold")
# print(baseline_optimized_cm)

# print("\nBalanced LightGBM, threshold 0.50")
# print(balanced_default_cm)

# print("\nBalanced LightGBM, optimized threshold")
# print(balanced_optimized_cm)

# print("\n5. CLASS-SPECIFIC PERFORMANCE")
# print("-" * 100)

# def class_metrics(actual, predicted, model_name):
#     cm = confusion_matrix(actual, predicted, labels=[-1, 1])

#     true_down, false_up = cm[0]
#     false_down, true_up = cm[1]

#     down_recall = true_down / (true_down + false_up)
#     up_recall = true_up / (true_up + false_down)

#     down_precision_denominator = true_down + false_down
#     up_precision_denominator = true_up + false_up

#     down_precision = (
#         true_down / down_precision_denominator
#         if down_precision_denominator > 0
#         else np.nan
#     )

#     up_precision = (
#         true_up / up_precision_denominator
#         if up_precision_denominator > 0
#         else np.nan
#     )

#     return {
#         "model": model_name,
#         "down_precision": down_precision,
#         "down_recall": down_recall,
#         "up_precision": up_precision,
#         "up_recall": up_recall,
#     }

# class_metric_rows = [
#     class_metrics(
#         y_valid,
#         lightgbm_valid_prediction,
#         "Baseline LightGBM 0.50",
#     ),
#     class_metrics(
#         y_valid,
#         lightgbm_optimized_prediction,
#         "Baseline LightGBM Optimized",
#     ),
#     class_metrics(
#         y_valid,
#         balanced_lightgbm_default_prediction,
#         "Balanced LightGBM 0.50",
#     ),
#     class_metrics(
#         y_valid,
#         balanced_lightgbm_optimized_prediction,
#         "Balanced LightGBM Optimized",
#     ),
# ]

# class_metrics_df = pd.DataFrame(class_metric_rows)

# print(class_metrics_df.to_string(index=False))

# print("\n6. PROBABILITY SUMMARIES")
# print("-" * 100)

# probability_summary = pd.DataFrame(
#     {
#         "baseline_lightgbm": pd.Series(lightgbm_valid_probability).describe(),
#         "balanced_lightgbm": pd.Series(
#             balanced_lightgbm_valid_probability
#         ).describe(),
#     }
# )

# print(probability_summary.to_string())

# print("\n7. TOP 10 THRESHOLDS — BASELINE LIGHTGBM")
# print("-" * 100)

# print(
#     lightgbm_threshold_results[
#         [
#             "threshold",
#             "direction_score",
#             "directional_return_pct",
#             "hit_rate",
#             "predicted_up_fraction",
#             "predicted_down_fraction",
#         ]
#     ]
#     .head(10)
#     .to_string(index=False)
# )

# print("\n8. TOP 10 THRESHOLDS — BALANCED LIGHTGBM")
# print("-" * 100)

# print(
#     balanced_threshold_results[
#         [
#             "threshold",
#             "direction_score",
#             "directional_return_pct",
#             "hit_rate",
#             "predicted_up_fraction",
#             "predicted_down_fraction",
#         ]
#     ]
#     .head(10)
#     .to_string(index=False)
# )

# print("\n" + "=" * 100)
# print("END OF DIRECTION MODEL UPDATE SUMMARY")
# print("=" * 100)

#### LightGBM Hyperparameter Search

This section tests a small number of LightGBM configurations.

For every configuration:

1. Train on the training set
2. Generate validation probabilities
3. Find the best validation threshold
4. Record validation performance

The test set remains unused.

In [ ]:
lightgbm_parameter_grid = [
    {
        "num_leaves": 15,
        "learning_rate": 0.05,
        "min_data_in_leaf": 100,
        "feature_fraction": 0.80,
        "bagging_fraction": 0.80,
    },
    {
        "num_leaves": 31,
        "learning_rate": 0.05,
        "min_data_in_leaf": 100,
        "feature_fraction": 0.80,
        "bagging_fraction": 0.80,
    },
    {
        "num_leaves": 63,
        "learning_rate": 0.05,
        "min_data_in_leaf": 100,
        "feature_fraction": 0.80,
        "bagging_fraction": 0.80,
    },
    {
        "num_leaves": 31,
        "learning_rate": 0.03,
        "min_data_in_leaf": 100,
        "feature_fraction": 0.80,
        "bagging_fraction": 0.80,
    },
    {
        "num_leaves": 31,
        "learning_rate": 0.08,
        "min_data_in_leaf": 100,
        "feature_fraction": 0.80,
        "bagging_fraction": 0.80,
    },
    {
        "num_leaves": 31,
        "learning_rate": 0.05,
        "min_data_in_leaf": 50,
        "feature_fraction": 0.80,
        "bagging_fraction": 0.80,
    },
    {
        "num_leaves": 31,
        "learning_rate": 0.05,
        "min_data_in_leaf": 250,
        "feature_fraction": 0.80,
        "bagging_fraction": 0.80,
    },
    {
        "num_leaves": 31,
        "learning_rate": 0.05,
        "min_data_in_leaf": 100,
        "feature_fraction": 1.00,
        "bagging_fraction": 0.80,
    },
    {
        "num_leaves": 31,
        "learning_rate": 0.05,
        "min_data_in_leaf": 100,
        "feature_fraction": 0.60,
        "bagging_fraction": 0.80,
    },
    {
        "num_leaves": 31,
        "learning_rate": 0.05,
        "min_data_in_leaf": 100,
        "feature_fraction": 0.80,
        "bagging_fraction": 1.00,
    },
]

print("Configurations to test:", len(lightgbm_parameter_grid))

In [ ]:
def run_lightgbm_configuration(configuration_id, parameter_changes):
    current_params = lightgbm_params.copy()
    current_params.update(parameter_changes)

    start_time = time.perf_counter()

    current_model = lgb.train(
        params=current_params,
        train_set=lightgbm_train_dataset,
        valid_sets=[lightgbm_valid_dataset],
        valid_names=["valid"],
        num_boost_round=1000,
        callbacks=[
            lgb.early_stopping(75, verbose=False),
            lgb.log_evaluation(0),
        ],
    )

    runtime_seconds = time.perf_counter() - start_time

    validation_probability = current_model.predict(
        X_valid_lgb,
        num_iteration=current_model.best_iteration,
    )

    threshold_results = evaluate_thresholds(
        model_name=f"LightGBM Config {configuration_id}",
        probability_up=validation_probability,
        actual_return_pct=valid_actual_return_pct,
        actual_direction=y_valid,
        thresholds=threshold_grid,
    )

    best_threshold_result = threshold_results.sort_values(
        "direction_score",
        ascending=False,
    ).iloc[0].to_dict()

    result = {
        "configuration_id": configuration_id,
        **parameter_changes,
        "best_iteration": current_model.best_iteration,
        "best_threshold": best_threshold_result["threshold"],
        "direction_score": best_threshold_result["direction_score"],
        "directional_return_pct": best_threshold_result["directional_return_pct"],
        "hit_rate": best_threshold_result["hit_rate"],
        "precision_up": best_threshold_result["precision_up"],
        "recall_up": best_threshold_result["recall_up"],
        "f1_up": best_threshold_result["f1_up"],
        "roc_auc": best_threshold_result["roc_auc"],
        "predicted_up_fraction": best_threshold_result["predicted_up_fraction"],
        "predicted_down_fraction": best_threshold_result["predicted_down_fraction"],
        "runtime_seconds": runtime_seconds,
    }

    return result, current_model, validation_probability

In [ ]:
lightgbm_tuning_results = []
lightgbm_tuning_models = {}
lightgbm_tuning_probabilities = {}

for configuration_id, parameter_changes in enumerate(lightgbm_parameter_grid, start=1):
    result, trained_model, validation_probability = run_lightgbm_configuration(
        configuration_id=configuration_id,
        parameter_changes=parameter_changes,
    )

    lightgbm_tuning_results.append(result)
    lightgbm_tuning_models[configuration_id] = trained_model
    lightgbm_tuning_probabilities[configuration_id] = validation_probability

    print(
        f"Completed {configuration_id}/{len(lightgbm_parameter_grid)} | "
        f"Score: {result['direction_score']:.6f} | "
        f"Threshold: {result['best_threshold']:.2f}"
    )

In [ ]:
lightgbm_tuning_results_df = pd.DataFrame(lightgbm_tuning_results)

lightgbm_tuning_results_df = lightgbm_tuning_results_df.sort_values(
    "direction_score",
    ascending=False,
).reset_index(drop=True)

lightgbm_tuning_results_df

In [ ]:
tuning_display_columns = [
    "configuration_id",
    "num_leaves",
    "learning_rate",
    "min_data_in_leaf",
    "feature_fraction",
    "bagging_fraction",
    "best_iteration",
    "best_threshold",
    "direction_score",
    "directional_return_pct",
    "hit_rate",
    "roc_auc",
    "predicted_up_fraction",
    "predicted_down_fraction",
    "runtime_seconds",
]

lightgbm_tuning_results_df[tuning_display_columns]

In [ ]:
best_tuning_row = lightgbm_tuning_results_df.iloc[0]

best_configuration_id = int(best_tuning_row["configuration_id"])
best_tuned_threshold = float(best_tuning_row["best_threshold"])
best_tuned_direction_score = float(best_tuning_row["direction_score"])

best_tuned_model = lightgbm_tuning_models[best_configuration_id]
best_tuned_valid_probability = lightgbm_tuning_probabilities[best_configuration_id]

print("Best configuration:", best_configuration_id)
print("Best threshold:", best_tuned_threshold)
print("Best direction score:", best_tuned_direction_score)
print("Best iteration:", best_tuned_model.best_iteration)

In [ ]:
best_tuned_valid_prediction = np.where(
    best_tuned_valid_probability >= best_tuned_threshold,
    1,
    -1,
)

In [ ]:
current_benchmark_score = 0.322316

score_improvement = best_tuned_direction_score - current_benchmark_score
relative_improvement_pct = score_improvement / current_benchmark_score * 100

print("Current benchmark score:", current_benchmark_score)
print("Best tuned score:", best_tuned_direction_score)
print("Absolute improvement:", score_improvement)
print("Relative improvement: {:.2f}%".format(relative_improvement_pct))

In [ ]:
tuned_vs_baseline_df = pd.DataFrame(
    [
        {
            "model": "Untuned LightGBM",
            "threshold": best_lightgbm_threshold,
            "direction_score": best_lightgbm_threshold_row["direction_score"],
            "directional_return_pct": best_lightgbm_threshold_row["directional_return_pct"],
            "hit_rate": best_lightgbm_threshold_row["hit_rate"],
            "roc_auc": best_lightgbm_threshold_row["roc_auc"],
            "predicted_up_fraction": best_lightgbm_threshold_row["predicted_up_fraction"],
            "predicted_down_fraction": best_lightgbm_threshold_row["predicted_down_fraction"],
        },
        {
            "model": "Tuned LightGBM",
            "threshold": best_tuning_row["best_threshold"],
            "direction_score": best_tuning_row["direction_score"],
            "directional_return_pct": best_tuning_row["directional_return_pct"],
            "hit_rate": best_tuning_row["hit_rate"],
            "roc_auc": best_tuning_row["roc_auc"],
            "predicted_up_fraction": best_tuning_row["predicted_up_fraction"],
            "predicted_down_fraction": best_tuning_row["predicted_down_fraction"],
        },
    ]
)

tuned_vs_baseline_df

In [ ]:
best_tuned_confusion_matrix = confusion_matrix(
    y_valid,
    best_tuned_valid_prediction,
    labels=[-1, 1],
)

best_tuned_confusion_matrix

In [ ]:
lightgbm_tuning_results_path = EXPERIMENTS_DIR / "direction_lightgbm_hyperparameter_search_v1.csv"

lightgbm_tuning_results_df.to_csv(
    lightgbm_tuning_results_path,
    index=False,
)

print("Saved:", lightgbm_tuning_results_path)

In [ ]:
best_direction_configuration = {
    "configuration_id": best_configuration_id,
    "parameters": {
        "num_leaves": int(best_tuning_row["num_leaves"]),
        "learning_rate": float(best_tuning_row["learning_rate"]),
        "min_data_in_leaf": int(best_tuning_row["min_data_in_leaf"]),
        "feature_fraction": float(best_tuning_row["feature_fraction"]),
        "bagging_fraction": float(best_tuning_row["bagging_fraction"]),
    },
    "best_iteration": int(best_tuning_row["best_iteration"]),
    "validation_threshold": best_tuned_threshold,
    "validation_direction_score": best_tuned_direction_score,
    "feature_set": "direction_features_v1",
    "seed": 42,
}

best_direction_configuration_path = CONFIGS_DIR / "direction_lightgbm_best_v1.json"

with open(best_direction_configuration_path, "w") as file:
    json.dump(best_direction_configuration, file, indent=4)

print("Saved:", best_direction_configuration_path)

In [ ]:
# print("=" * 100)
# print("LIGHTGBM HYPERPARAMETER TUNING SUMMARY")
# print("=" * 100)

# print("\n1. BEST CONFIGURATION")
# print("-" * 100)
# print(best_tuning_row)

# print("\n2. TOP 10 CONFIGURATIONS")
# print("-" * 100)

# display_columns = [
#     "configuration_id",
#     "num_leaves",
#     "learning_rate",
#     "min_data_in_leaf",
#     "feature_fraction",
#     "bagging_fraction",
#     "best_iteration",
#     "best_threshold",
#     "direction_score",
#     "directional_return_pct",
#     "hit_rate",
#     "roc_auc",
# ]

# print(
#     lightgbm_tuning_results_df[display_columns]
#     .head(10)
#     .to_string(index=False)
# )

# print("\n3. BASELINE VS BEST TUNED")
# print("-" * 100)

# print(tuned_vs_baseline_df.to_string(index=False))

# print("\n4. SCORE IMPROVEMENT")
# print("-" * 100)

# print(f"Current Benchmark Direction Score : {current_benchmark_score:.6f}")
# print(f"Best Tuned Direction Score        : {best_tuned_direction_score:.6f}")
# print(f"Absolute Improvement             : {score_improvement:.6f}")
# print(f"Relative Improvement             : {relative_improvement_pct:.2f}%")

# print("\n5. CONFUSION MATRIX OF BEST MODEL")
# print("-" * 100)

# print(best_tuned_confusion_matrix)

# print("\n6. BEST CONFIGURATION FILE")
# print("-" * 100)

# print(best_direction_configuration)

# print("\n" + "=" * 100)
# print("END OF SUMMARY")
# print("=" * 100)

#### Direction Feature Engineering — Version 2

New candidate features:

- Momentum acceleration
- Relative return versus universe
- Bollinger z-score
- ADX
- Supertrend (7, 1)
- Distance from rolling high and low
- Positive-day fraction
- Consecutive return streak

All features use information available by the close of `pred_date`.

In [ ]:
direction_v2_df = model_df.copy()

direction_v2_df = direction_v2_df.sort_values(["symbol", "pred_date"]).reset_index(drop=True)

print(direction_v2_df.shape)


In [ ]:
direction_v2_df["momentum_acceleration_5d_20d"] = (
    direction_v2_df["return_5d"] - direction_v2_df["return_20d"] / 4
)

In [ ]:
direction_v2_df["close_mean_20d"] = (
    direction_v2_df.groupby("symbol")["close"]
    .transform(lambda series: series.rolling(20, min_periods=20).mean())
)

direction_v2_df["close_std_20d"] = (
    direction_v2_df.groupby("symbol")["close"]
    .transform(lambda series: series.rolling(20, min_periods=20).std())
)

direction_v2_df["bollinger_zscore_20d"] = (
    (direction_v2_df["close"] - direction_v2_df["close_mean_20d"])
    / direction_v2_df["close_std_20d"]
)

In [ ]:
direction_v2_df["rolling_high_20d"] = (
    direction_v2_df.groupby("symbol")["high"]
    .transform(lambda series: series.rolling(20, min_periods=20).max())
)

direction_v2_df["rolling_low_20d"] = (
    direction_v2_df.groupby("symbol")["low"]
    .transform(lambda series: series.rolling(20, min_periods=20).min())
)

direction_v2_df["distance_from_20d_high_pct"] = (
    direction_v2_df["close"] / direction_v2_df["rolling_high_20d"] - 1
) * 100

direction_v2_df["distance_from_20d_low_pct"] = (
    direction_v2_df["close"] / direction_v2_df["rolling_low_20d"] - 1
) * 100

In [ ]:
direction_v2_df["positive_day_fraction_5d"] = (
    direction_v2_df.groupby("symbol")["return_1d"]
    .transform(lambda series: series.gt(0).rolling(5, min_periods=5).mean())
)

direction_v2_df["positive_day_fraction_20d"] = (
    direction_v2_df.groupby("symbol")["return_1d"]
    .transform(lambda series: series.gt(0).rolling(20, min_periods=20).mean())
)

In [ ]:
def calculate_signed_streak(series):
    signs = np.sign(series.fillna(0)).astype(int)

    streak_values = []
    current_streak = 0
    previous_sign = 0

    for current_sign in signs:
        if current_sign == 0:
            current_streak = 0
        elif current_sign == previous_sign:
            current_streak += current_sign
        else:
            current_streak = current_sign

        streak_values.append(current_streak)
        previous_sign = current_sign

    return pd.Series(streak_values, index=series.index)

In [ ]:
direction_v2_df["return_streak"] = (
    direction_v2_df.groupby("symbol", group_keys=False)["return_1d"]
    .apply(calculate_signed_streak)
)

In [ ]:
previous_close = direction_v2_df.groupby("symbol")["close"].shift(1)

true_range_components = pd.concat(
    [
        direction_v2_df["high"] - direction_v2_df["low"],
        (direction_v2_df["high"] - previous_close).abs(),
        (direction_v2_df["low"] - previous_close).abs(),
    ],
    axis=1,
)

direction_v2_df["true_range"] = true_range_components.max(axis=1)

In [ ]:
up_move = direction_v2_df.groupby("symbol")["high"].diff()
down_move = -direction_v2_df.groupby("symbol")["low"].diff()

direction_v2_df["plus_dm"] = np.where(
    (up_move > down_move) & (up_move > 0),
    up_move,
    0,
)

direction_v2_df["minus_dm"] = np.where(
    (down_move > up_move) & (down_move > 0),
    down_move,
    0,
)

In [ ]:
direction_v2_df["atr_14"] = (
    direction_v2_df.groupby("symbol")["true_range"]
    .transform(lambda series: series.ewm(alpha=1 / 14, adjust=False, min_periods=14).mean())
)

direction_v2_df["plus_dm_smoothed_14"] = (
    direction_v2_df.groupby("symbol")["plus_dm"]
    .transform(lambda series: series.ewm(alpha=1 / 14, adjust=False, min_periods=14).mean())
)

direction_v2_df["minus_dm_smoothed_14"] = (
    direction_v2_df.groupby("symbol")["minus_dm"]
    .transform(lambda series: series.ewm(alpha=1 / 14, adjust=False, min_periods=14).mean())
)

direction_v2_df["plus_di_14"] = (
    100 * direction_v2_df["plus_dm_smoothed_14"] / direction_v2_df["atr_14"]
)

direction_v2_df["minus_di_14"] = (
    100 * direction_v2_df["minus_dm_smoothed_14"] / direction_v2_df["atr_14"]
)

direction_v2_df["dx_14"] = (
    100
    * (direction_v2_df["plus_di_14"] - direction_v2_df["minus_di_14"]).abs()
    / (direction_v2_df["plus_di_14"] + direction_v2_df["minus_di_14"])
)

direction_v2_df["adx_14"] = (
    direction_v2_df.groupby("symbol")["dx_14"]
    .transform(lambda series: series.ewm(alpha=1 / 14, adjust=False, min_periods=14).mean())
)

In [ ]:
direction_v2_df["di_spread_14"] = (
    direction_v2_df["plus_di_14"] - direction_v2_df["minus_di_14"]
)

In [ ]:
def calculate_supertrend(group, period=7, multiplier=1.0):
    group = group.sort_values("pred_date").copy()

    high = group["high"].to_numpy(dtype=float)
    low = group["low"].to_numpy(dtype=float)
    close = group["close"].to_numpy(dtype=float)

    n = len(group)

    true_range = np.full(n, np.nan)
    atr = np.full(n, np.nan)
    basic_upper = np.full(n, np.nan)
    basic_lower = np.full(n, np.nan)
    final_upper = np.full(n, np.nan)
    final_lower = np.full(n, np.nan)
    supertrend = np.full(n, np.nan)
    direction = np.full(n, np.nan)

    if n == 0:
        return group

    true_range[0] = high[0] - low[0]

    for i in range(1, n):
        true_range[i] = max(
            high[i] - low[i],
            abs(high[i] - close[i - 1]),
            abs(low[i] - close[i - 1]),
        )

    if n >= period:
        atr[period - 1] = np.nanmean(true_range[:period])

        for i in range(period, n):
            atr[i] = (
                atr[i - 1] * (period - 1) + true_range[i]
            ) / period

    midpoint = (high + low) / 2

    basic_upper = midpoint + multiplier * atr
    basic_lower = midpoint - multiplier * atr

    start_index = period - 1

    if n > start_index and not np.isnan(atr[start_index]):
        final_upper[start_index] = basic_upper[start_index]
        final_lower[start_index] = basic_lower[start_index]

        if close[start_index] >= midpoint[start_index]:
            supertrend[start_index] = final_lower[start_index]
            direction[start_index] = 1
        else:
            supertrend[start_index] = final_upper[start_index]
            direction[start_index] = -1

        for i in range(start_index + 1, n):
            if (
                basic_upper[i] < final_upper[i - 1]
                or close[i - 1] > final_upper[i - 1]
            ):
                final_upper[i] = basic_upper[i]
            else:
                final_upper[i] = final_upper[i - 1]

            if (
                basic_lower[i] > final_lower[i - 1]
                or close[i - 1] < final_lower[i - 1]
            ):
                final_lower[i] = basic_lower[i]
            else:
                final_lower[i] = final_lower[i - 1]

            if supertrend[i - 1] == final_upper[i - 1]:
                if close[i] <= final_upper[i]:
                    supertrend[i] = final_upper[i]
                    direction[i] = -1
                else:
                    supertrend[i] = final_lower[i]
                    direction[i] = 1
            else:
                if close[i] >= final_lower[i]:
                    supertrend[i] = final_lower[i]
                    direction[i] = 1
                else:
                    supertrend[i] = final_upper[i]
                    direction[i] = -1

    group["supertrend_7_1"] = supertrend
    group["supertrend_direction_7_1"] = direction

    group["supertrend_distance_pct_7_1"] = (
        group["close"] / group["supertrend_7_1"] - 1
    ) * 100

    return group

In [ ]:
direction_v2_df = (
    direction_v2_df.groupby("symbol", group_keys=False)
    .apply(
        calculate_supertrend,
        period=7,
        multiplier=1.0,
    )
    .reset_index(drop=True)
)

direction_v2_df = direction_v2_df.sort_values(
    ["symbol", "pred_date"]
).reset_index(drop=True)

print("Supertrend recalculated.")

In [ ]:
direction_v2_feature_columns = [
    "momentum_acceleration_5d_20d",
    "relative_return_1d",
    "bollinger_zscore_20d",
    "distance_from_20d_high_pct",
    "distance_from_20d_low_pct",
    "positive_day_fraction_5d",
    "positive_day_fraction_20d",
    "return_streak",
    "adx_14",
    "di_spread_14",
    "supertrend_direction_7_1",
    "supertrend_distance_pct_7_1",
]

direction_feature_columns_v2 = (
    base_numeric_feature_columns
    + direction_v2_feature_columns
)

print("Base features:", len(base_numeric_feature_columns))
print("New V2 features:", len(direction_v2_feature_columns))
print("Total V2 numeric features:", len(direction_feature_columns_v2))

In [ ]:
missing_v2_columns = [
    column
    for column in direction_v2_feature_columns
    if column not in direction_v2_df.columns
]

print("Missing V2 columns:", missing_v2_columns)

In [ ]:
direction_v2_df["relative_return_1d"] = (
    direction_v2_df["return_1d"]
    - direction_v2_df["universe_mean_return_1d"]
)

In [ ]:
direction_v2_df[direction_v2_feature_columns] = (
    direction_v2_df[direction_v2_feature_columns]
    .replace([np.inf, -np.inf], np.nan)
)

In [ ]:
v2_missingness = (
    direction_v2_df[direction_v2_feature_columns]
    .isna()
    .sum()
    .to_frame("missing_count")
)

v2_missingness["missing_pct"] = (
    v2_missingness["missing_count"] / len(direction_v2_df) * 100
)

v2_missingness.sort_values("missing_pct", ascending=False)

In [ ]:
assert direction_v2_df["positive_day_fraction_5d"].dropna().between(0, 1).all()
assert direction_v2_df["positive_day_fraction_20d"].dropna().between(0, 1).all()
assert direction_v2_df["supertrend_direction_7_1"].dropna().isin([-1, 1]).all()
assert (direction_v2_df["adx_14"].dropna() >= 0).all()

print("Direction V2 feature checks passed.")

#### Direction Feature Set V2 — Model Test

The model architecture, hyperparameters, training period and validation period remain unchanged.

Only the feature set changes:

- V1: 25 numeric features + symbol
- V2: V1 features + 12 new direction-specific features

In [ ]:
direction_v2_model_df = direction_v2_df.loc[
    direction_v2_df["split"].isin(["train", "valid", "test"])
].copy()

direction_v2_model_df = direction_v2_model_df.dropna(
    subset=daily_feature_columns
).reset_index(drop=True)

print("Rows available:", len(direction_v2_model_df))

In [ ]:
train_v2_df = direction_v2_model_df.loc[
    direction_v2_model_df["split"] == "train"
].copy()

valid_v2_df = direction_v2_model_df.loc[
    direction_v2_model_df["split"] == "valid"
].copy()

test_v2_df = direction_v2_model_df.loc[
    direction_v2_model_df["split"] == "test"
].copy()

print("Train rows:", len(train_v2_df))
print("Validation rows:", len(valid_v2_df))
print("Test rows:", len(test_v2_df))

In [ ]:
comparison_keys = [
    "symbol",
    "pred_date",
    "actual_direction",
    "split",
]

v1_target_check = direction_df.loc[
    direction_df["split"].isin(["train", "valid", "test"])
].copy()

v1_target_check = v1_target_check.dropna(
    subset=daily_feature_columns
)

v1_target_check = v1_target_check[
    comparison_keys
].sort_values(
    ["split", "symbol", "pred_date"]
).reset_index(drop=True)

v2_target_check = direction_v2_model_df[
    comparison_keys
].sort_values(
    ["split", "symbol", "pred_date"]
).reset_index(drop=True)

assert len(v1_target_check) == len(v2_target_check)

assert v1_target_check[
    ["split", "symbol", "pred_date"]
].equals(
    v2_target_check[
        ["split", "symbol", "pred_date"]
    ]
)

assert np.array_equal(
    v1_target_check["actual_direction"].to_numpy(),
    v2_target_check["actual_direction"].to_numpy(),
)

print("V1 and V2 rows and targets are aligned.")

In [ ]:
train_v2_df = direction_v2_model_df.loc[
    direction_v2_model_df["split"] == "train"
].sort_values(
    ["symbol", "pred_date"]
).reset_index(drop=True)

valid_v2_df = direction_v2_model_df.loc[
    direction_v2_model_df["split"] == "valid"
].sort_values(
    ["symbol", "pred_date"]
).reset_index(drop=True)

test_v2_df = direction_v2_model_df.loc[
    direction_v2_model_df["split"] == "test"
].sort_values(
    ["symbol", "pred_date"]
).reset_index(drop=True)

y_train_v2 = train_v2_df["actual_direction"].astype("int8").reset_index(drop=True)
y_valid_v2 = valid_v2_df["actual_direction"].astype("int8").reset_index(drop=True)
y_test_v2 = test_v2_df["actual_direction"].astype("int8").reset_index(drop=True)

y_train_v2_binary = (y_train_v2 == 1).astype("int8")
y_valid_v2_binary = (y_valid_v2 == 1).astype("int8")
y_test_v2_binary = (y_test_v2 == 1).astype("int8")

valid_v2_actual_return_pct = valid_v2_df["actual_return_pct"].reset_index(drop=True)

print("Train rows:", len(train_v2_df))
print("Validation rows:", len(valid_v2_df))
print("Test rows:", len(test_v2_df))

In [ ]:
lightgbm_feature_columns_v2 = direction_feature_columns_v2 + ["symbol"]

X_train_lgb_v2 = train_v2_df[lightgbm_feature_columns_v2].copy()
X_valid_lgb_v2 = valid_v2_df[lightgbm_feature_columns_v2].copy()
X_test_lgb_v2 = test_v2_df[lightgbm_feature_columns_v2].copy()

X_train_lgb_v2["symbol"] = X_train_lgb_v2["symbol"].astype(str).astype(symbol_dtype)
X_valid_lgb_v2["symbol"] = X_valid_lgb_v2["symbol"].astype(str).astype(symbol_dtype)
X_test_lgb_v2["symbol"] = X_test_lgb_v2["symbol"].astype(str).astype(symbol_dtype)

print("V2 feature count:", len(lightgbm_feature_columns_v2))
print("Train matrix:", X_train_lgb_v2.shape)
print("Validation matrix:", X_valid_lgb_v2.shape)
print("Test matrix:", X_test_lgb_v2.shape)

In [ ]:
train_v2_df = direction_v2_model_df.loc[
    direction_v2_model_df["split"] == "train"
].sort_values(
    ["symbol", "pred_date"]
).reset_index(drop=True)

valid_v2_df = direction_v2_model_df.loc[
    direction_v2_model_df["split"] == "valid"
].sort_values(
    ["symbol", "pred_date"]
).reset_index(drop=True)

test_v2_df = direction_v2_model_df.loc[
    direction_v2_model_df["split"] == "test"
].sort_values(
    ["symbol", "pred_date"]
).reset_index(drop=True)

print("Train rows:", len(train_v2_df))
print("Validation rows:", len(valid_v2_df))
print("Test rows:", len(test_v2_df))

In [ ]:
y_train_v2 = train_v2_df["actual_direction"].astype("int8").reset_index(drop=True)
y_valid_v2 = valid_v2_df["actual_direction"].astype("int8").reset_index(drop=True)
y_test_v2 = test_v2_df["actual_direction"].astype("int8").reset_index(drop=True)

y_train_v2_binary = (y_train_v2 == 1).astype("int8")
y_valid_v2_binary = (y_valid_v2 == 1).astype("int8")
y_test_v2_binary = (y_test_v2 == 1).astype("int8")

valid_v2_actual_return_pct = valid_v2_df["actual_return_pct"].reset_index(drop=True)

print("V2 targets created.")

In [ ]:
assert len(train_v2_df) == len(train_df)
assert len(valid_v2_df) == len(valid_df)
assert len(test_v2_df) == len(test_df)

assert train_v2_df[["symbol", "pred_date"]].equals(
    train_df.sort_values(["symbol", "pred_date"])[["symbol", "pred_date"]].reset_index(drop=True)
)

assert valid_v2_df[["symbol", "pred_date"]].equals(
    valid_df.sort_values(["symbol", "pred_date"])[["symbol", "pred_date"]].reset_index(drop=True)
)

assert test_v2_df[["symbol", "pred_date"]].equals(
    test_df.sort_values(["symbol", "pred_date"])[["symbol", "pred_date"]].reset_index(drop=True)
)

print("V1 and V2 split rows are aligned.")

In [ ]:
lightgbm_feature_columns_v2 = direction_feature_columns_v2 + ["symbol"]

print("V2 numeric features:", len(direction_feature_columns_v2))
print("V2 total features including symbol:", len(lightgbm_feature_columns_v2))

In [ ]:
X_train_lgb_v2 = train_v2_df[lightgbm_feature_columns_v2].copy().reset_index(drop=True)
X_valid_lgb_v2 = valid_v2_df[lightgbm_feature_columns_v2].copy().reset_index(drop=True)
X_test_lgb_v2 = test_v2_df[lightgbm_feature_columns_v2].copy().reset_index(drop=True)

X_train_lgb_v2["symbol"] = X_train_lgb_v2["symbol"].astype(str).astype(symbol_dtype)
X_valid_lgb_v2["symbol"] = X_valid_lgb_v2["symbol"].astype(str).astype(symbol_dtype)
X_test_lgb_v2["symbol"] = X_test_lgb_v2["symbol"].astype(str).astype(symbol_dtype)

print("Train matrix:", X_train_lgb_v2.shape)
print("Validation matrix:", X_valid_lgb_v2.shape)
print("Test matrix:", X_test_lgb_v2.shape)
print("Symbol dtype:", X_train_lgb_v2["symbol"].dtype)

In [ ]:
v2_model_feature_missingness = pd.DataFrame(
    {
        "train_missing_pct": X_train_lgb_v2[direction_v2_feature_columns].isna().mean() * 100,
        "valid_missing_pct": X_valid_lgb_v2[direction_v2_feature_columns].isna().mean() * 100,
        "test_missing_pct": X_test_lgb_v2[direction_v2_feature_columns].isna().mean() * 100,
    }
)

v2_model_feature_missingness.sort_values(
    "valid_missing_pct",
    ascending=False,
)

In [ ]:
direction_v2_df[
    [
        "symbol",
        "pred_date",
        "close",
        "supertrend_7_1",
        "supertrend_direction_7_1",
        "supertrend_distance_pct_7_1",
    ]
].tail(20)

In [ ]:
direction_v2_df[
    [
        "supertrend_7_1",
        "supertrend_direction_7_1",
        "supertrend_distance_pct_7_1",
    ]
].describe(include="all")

In [ ]:
missing_v2_columns = [
    column
    for column in direction_v2_feature_columns
    if column not in direction_v2_df.columns
]

assert not missing_v2_columns, f"Missing V2 columns: {missing_v2_columns}"

direction_v2_df[direction_v2_feature_columns] = (
    direction_v2_df[direction_v2_feature_columns]
    .replace([np.inf, -np.inf], np.nan)
)

print("All V2 features are present.")

In [ ]:
assert direction_v2_df["positive_day_fraction_5d"].dropna().between(0, 1).all()
assert direction_v2_df["positive_day_fraction_20d"].dropna().between(0, 1).all()
assert direction_v2_df["supertrend_direction_7_1"].dropna().isin([-1, 1]).all()
assert (direction_v2_df["adx_14"].dropna() >= 0).all()

duplicate_v2_keys = direction_v2_df.duplicated(
    subset=["symbol", "pred_date"]
).sum()

assert duplicate_v2_keys == 0

print("Direction V2 feature sanity checks passed.")

In [ ]:
v2_missingness = direction_v2_df[
    direction_v2_feature_columns
].isna().sum().to_frame("missing_count")

v2_missingness["missing_pct"] = (
    v2_missingness["missing_count"]
    / len(direction_v2_df)
    * 100
)

v2_missingness.sort_values(
    "missing_pct",
    ascending=False,
)

In [ ]:
direction_v2_model_df = direction_v2_df.loc[
    direction_v2_df["split"].isin(["train", "valid", "test"])
].copy()

direction_v2_model_df = direction_v2_model_df.dropna(
    subset=daily_feature_columns
)

direction_v2_model_df = direction_v2_model_df.sort_values(
    ["split", "symbol", "pred_date"]
).reset_index(drop=True)

print("V2 modelling rows:", len(direction_v2_model_df))

In [ ]:
comparison_columns = [
    "split",
    "symbol",
    "pred_date",
    "actual_direction",
    "actual_return_pct",
]

v1_target_check = direction_df.loc[
    direction_df["split"].isin(["train", "valid", "test"])
].copy()

v1_target_check = v1_target_check.dropna(
    subset=daily_feature_columns
)

v1_target_check = v1_target_check[
    comparison_columns
].sort_values(
    ["split", "symbol", "pred_date"]
).reset_index(drop=True)

v2_target_check = direction_v2_model_df[
    comparison_columns
].sort_values(
    ["split", "symbol", "pred_date"]
).reset_index(drop=True)

assert len(v1_target_check) == len(v2_target_check)

assert v1_target_check[
    ["split", "symbol", "pred_date"]
].equals(
    v2_target_check[
        ["split", "symbol", "pred_date"]
    ]
)

assert np.array_equal(
    v1_target_check["actual_direction"].to_numpy(),
    v2_target_check["actual_direction"].to_numpy(),
)

assert np.allclose(
    v1_target_check["actual_return_pct"].to_numpy(),
    v2_target_check["actual_return_pct"].to_numpy(),
    equal_nan=True,
)

print("V1 and V2 rows and targets are aligned.")

In [ ]:
train_v2_df = direction_v2_model_df.loc[
    direction_v2_model_df["split"] == "train"
].sort_values(
    ["symbol", "pred_date"]
).reset_index(drop=True)

valid_v2_df = direction_v2_model_df.loc[
    direction_v2_model_df["split"] == "valid"
].sort_values(
    ["symbol", "pred_date"]
).reset_index(drop=True)

test_v2_df = direction_v2_model_df.loc[
    direction_v2_model_df["split"] == "test"
].sort_values(
    ["symbol", "pred_date"]
).reset_index(drop=True)

print("Train rows:", len(train_v2_df))
print("Validation rows:", len(valid_v2_df))
print("Test rows:", len(test_v2_df))

In [ ]:
y_train_v2 = train_v2_df[
    "actual_direction"
].astype("int8").reset_index(drop=True)

y_valid_v2 = valid_v2_df[
    "actual_direction"
].astype("int8").reset_index(drop=True)

y_test_v2 = test_v2_df[
    "actual_direction"
].astype("int8").reset_index(drop=True)

y_train_v2_binary = (y_train_v2 == 1).astype("int8")
y_valid_v2_binary = (y_valid_v2 == 1).astype("int8")
y_test_v2_binary = (y_test_v2 == 1).astype("int8")

valid_v2_actual_return_pct = valid_v2_df[
    "actual_return_pct"
].reset_index(drop=True)

print("V2 targets created.")

In [ ]:
train_v1_sorted = train_df.sort_values(
    ["symbol", "pred_date"]
).reset_index(drop=True)

valid_v1_sorted = valid_df.sort_values(
    ["symbol", "pred_date"]
).reset_index(drop=True)

test_v1_sorted = test_df.sort_values(
    ["symbol", "pred_date"]
).reset_index(drop=True)

assert train_v2_df[["symbol", "pred_date"]].equals(
    train_v1_sorted[["symbol", "pred_date"]]
)

assert valid_v2_df[["symbol", "pred_date"]].equals(
    valid_v1_sorted[["symbol", "pred_date"]]
)

assert test_v2_df[["symbol", "pred_date"]].equals(
    test_v1_sorted[["symbol", "pred_date"]]
)

print("V1 and V2 split rows match exactly.")

In [ ]:
direction_feature_columns_v2 = (
    base_numeric_feature_columns
    + direction_v2_feature_columns
)

lightgbm_feature_columns_v2 = (
    direction_feature_columns_v2
    + ["symbol"]
)

print("V1 numeric features:", len(base_numeric_feature_columns))
print("New V2 features:", len(direction_v2_feature_columns))
print("V2 numeric features:", len(direction_feature_columns_v2))
print("Total features including symbol:", len(lightgbm_feature_columns_v2))

In [ ]:
X_train_lgb_v2 = train_v2_df[
    lightgbm_feature_columns_v2
].copy().reset_index(drop=True)

X_valid_lgb_v2 = valid_v2_df[
    lightgbm_feature_columns_v2
].copy().reset_index(drop=True)

X_test_lgb_v2 = test_v2_df[
    lightgbm_feature_columns_v2
].copy().reset_index(drop=True)

X_train_lgb_v2["symbol"] = (
    X_train_lgb_v2["symbol"]
    .astype(str)
    .astype(symbol_dtype)
)

X_valid_lgb_v2["symbol"] = (
    X_valid_lgb_v2["symbol"]
    .astype(str)
    .astype(symbol_dtype)
)

X_test_lgb_v2["symbol"] = (
    X_test_lgb_v2["symbol"]
    .astype(str)
    .astype(symbol_dtype)
)

print("Train matrix:", X_train_lgb_v2.shape)
print("Validation matrix:", X_valid_lgb_v2.shape)
print("Test matrix:", X_test_lgb_v2.shape)

In [ ]:
v2_model_feature_missingness = pd.DataFrame(
    {
        "train_missing_pct": (
            X_train_lgb_v2[direction_v2_feature_columns]
            .isna()
            .mean()
            * 100
        ),
        "valid_missing_pct": (
            X_valid_lgb_v2[direction_v2_feature_columns]
            .isna()
            .mean()
            * 100
        ),
        "test_missing_pct": (
            X_test_lgb_v2[direction_v2_feature_columns]
            .isna()
            .mean()
            * 100
        ),
    }
)

v2_model_feature_missingness.sort_values(
    "valid_missing_pct",
    ascending=False,
)

In [ ]:
lightgbm_train_dataset_v2 = lgb.Dataset(
    X_train_lgb_v2,
    label=y_train_v2_binary,
    categorical_feature=["symbol"],
    free_raw_data=False,
)

lightgbm_valid_dataset_v2 = lgb.Dataset(
    X_valid_lgb_v2,
    label=y_valid_v2_binary,
    categorical_feature=["symbol"],
    free_raw_data=False,
)

In [ ]:
direction_v2_params = lightgbm_params.copy()

direction_v2_params.update(
    {
        "num_leaves": 31,
        "learning_rate": 0.05,
        "min_data_in_leaf": 100,
        "feature_fraction": 0.60,
        "bagging_fraction": 0.80,
    }
)

direction_v2_params

In [ ]:
direction_v2_start_time = time.perf_counter()

direction_v2_model = lgb.train(
    params=direction_v2_params,
    train_set=lightgbm_train_dataset_v2,
    valid_sets=[lightgbm_valid_dataset_v2],
    valid_names=["valid"],
    num_boost_round=1000,
    callbacks=[
        lgb.early_stopping(75, verbose=False),
        lgb.log_evaluation(0),
    ],
)

direction_v2_runtime = (
    time.perf_counter()
    - direction_v2_start_time
)

print(f"Training time: {direction_v2_runtime:.2f} seconds")
print("Best iteration:", direction_v2_model.best_iteration)

In [ ]:
direction_v2_valid_probability = direction_v2_model.predict(
    X_valid_lgb_v2,
    num_iteration=direction_v2_model.best_iteration,
)

pd.Series(
    direction_v2_valid_probability
).describe()

In [ ]:
direction_v2_threshold_results = evaluate_thresholds(
    model_name="LightGBM Direction Features V2",
    probability_up=direction_v2_valid_probability,
    actual_return_pct=valid_v2_actual_return_pct,
    actual_direction=y_valid_v2,
    thresholds=threshold_grid,
)

direction_v2_threshold_results = (
    direction_v2_threshold_results
    .sort_values(
        "direction_score",
        ascending=False,
    )
    .reset_index(drop=True)
)

direction_v2_threshold_results.head(10)

In [ ]:
best_direction_v2_row = direction_v2_threshold_results.iloc[0]

best_direction_v2_threshold = float(
    best_direction_v2_row["threshold"]
)

best_direction_v2_score = float(
    best_direction_v2_row["direction_score"]
)

print("Best V2 threshold:", best_direction_v2_threshold)
print("Best V2 direction score:", best_direction_v2_score)
print(
    "Best V2 directional return:",
    best_direction_v2_row["directional_return_pct"],
)
print("Best V2 hit rate:", best_direction_v2_row["hit_rate"])
print(
    "Predicted up fraction:",
    best_direction_v2_row["predicted_up_fraction"],
)
print(
    "Predicted down fraction:",
    best_direction_v2_row["predicted_down_fraction"],
)

In [ ]:
direction_v2_valid_prediction = np.where(
    direction_v2_valid_probability
    >= best_direction_v2_threshold,
    1,
    -1,
)

In [ ]:
direction_feature_comparison_df = pd.DataFrame(
    [
        {
            "feature_set": "Direction V1",
            "numeric_features": len(
                base_numeric_feature_columns
            ),
            "best_iteration": int(
                best_tuned_model.best_iteration
            ),
            "threshold": best_tuned_threshold,
            "direction_score": best_tuned_direction_score,
            "directional_return_pct": float(
                best_tuning_row[
                    "directional_return_pct"
                ]
            ),
            "hit_rate": float(
                best_tuning_row["hit_rate"]
            ),
            "roc_auc": float(
                best_tuning_row["roc_auc"]
            ),
            "predicted_up_fraction": float(
                best_tuning_row[
                    "predicted_up_fraction"
                ]
            ),
            "predicted_down_fraction": float(
                best_tuning_row[
                    "predicted_down_fraction"
                ]
            ),
        },
        {
            "feature_set": "Direction V2",
            "numeric_features": len(
                direction_feature_columns_v2
            ),
            "best_iteration": int(
                direction_v2_model.best_iteration
            ),
            "threshold": best_direction_v2_threshold,
            "direction_score": best_direction_v2_score,
            "directional_return_pct": float(
                best_direction_v2_row[
                    "directional_return_pct"
                ]
            ),
            "hit_rate": float(
                best_direction_v2_row["hit_rate"]
            ),
            "roc_auc": float(
                best_direction_v2_row["roc_auc"]
            ),
            "predicted_up_fraction": float(
                best_direction_v2_row[
                    "predicted_up_fraction"
                ]
            ),
            "predicted_down_fraction": float(
                best_direction_v2_row[
                    "predicted_down_fraction"
                ]
            ),
        },
    ]
)

direction_feature_comparison_df

In [ ]:
v2_absolute_improvement = (
    best_direction_v2_score
    - best_tuned_direction_score
)

v2_relative_improvement_pct = (
    v2_absolute_improvement
    / best_tuned_direction_score
    * 100
)

print("V1 direction score:", best_tuned_direction_score)
print("V2 direction score:", best_direction_v2_score)
print("Absolute improvement:", v2_absolute_improvement)
print(
    "Relative improvement: {:.2f}%".format(
        v2_relative_improvement_pct
    )
)

In [ ]:
direction_v2_confusion_matrix = confusion_matrix(
    y_valid_v2,
    direction_v2_valid_prediction,
    labels=[-1, 1],
)

direction_v2_confusion_matrix

In [ ]:
direction_v2_feature_importance = pd.DataFrame(
    {
        "feature": direction_v2_model.feature_name(),
        "gain_importance": (
            direction_v2_model.feature_importance(
                importance_type="gain"
            )
        ),
        "split_importance": (
            direction_v2_model.feature_importance(
                importance_type="split"
            )
        ),
    }
)

direction_v2_feature_importance = (
    direction_v2_feature_importance
    .sort_values(
        "gain_importance",
        ascending=False,
    )
    .reset_index(drop=True)
)

direction_v2_feature_importance.head(5)

In [ ]:
new_v2_feature_importance = (
    direction_v2_feature_importance.loc[
        direction_v2_feature_importance[
            "feature"
        ].isin(direction_v2_feature_columns)
    ]
    .copy()
    .reset_index(drop=True)
)

new_v2_feature_importance

In [ ]:
direction_feature_comparison_path = (
    EXPERIMENTS_DIR
    / "direction_feature_set_comparison_v2.csv"
)

direction_v2_threshold_path = (
    EXPERIMENTS_DIR
    / "direction_v2_threshold_search.csv"
)

direction_v2_importance_path = (
    EXPERIMENTS_DIR
    / "direction_v2_feature_importance.csv"
)

direction_v2_new_importance_path = (
    EXPERIMENTS_DIR
    / "direction_v2_new_feature_importance.csv"
)

direction_feature_comparison_df.to_csv(
    direction_feature_comparison_path,
    index=False,
)

direction_v2_threshold_results.to_csv(
    direction_v2_threshold_path,
    index=False,
)

direction_v2_feature_importance.to_csv(
    direction_v2_importance_path,
    index=False,
)

new_v2_feature_importance.to_csv(
    direction_v2_new_importance_path,
    index=False,
)

print("Saved:", direction_feature_comparison_path)
print("Saved:", direction_v2_threshold_path)
print("Saved:", direction_v2_importance_path)
print("Saved:", direction_v2_new_importance_path)

In [ ]:
best_direction_v2_configuration = {
    "feature_set": "direction_features_v2",
    "numeric_feature_count": len(
        direction_feature_columns_v2
    ),
    "features": direction_feature_columns_v2,
    "parameters": direction_v2_params,
    "best_iteration": int(
        direction_v2_model.best_iteration
    ),
    "validation_threshold": (
        best_direction_v2_threshold
    ),
    "validation_direction_score": (
        best_direction_v2_score
    ),
    "validation_directional_return_pct": float(
        best_direction_v2_row[
            "directional_return_pct"
        ]
    ),
    "validation_hit_rate": float(
        best_direction_v2_row["hit_rate"]
    ),
    "seed": 42,
}

best_direction_v2_configuration_path = (
    CONFIGS_DIR
    / "direction_lightgbm_v2.json"
)

with open(
    best_direction_v2_configuration_path,
    "w",
) as file:
    json.dump(
        best_direction_v2_configuration,
        file,
        indent=4,
    )

print(
    "Saved:",
    best_direction_v2_configuration_path,
)

In [ ]:
print("=" * 100)
print("DIRECTION FEATURE SET V2 SUMMARY")
print("=" * 100)

print("\n1. V1 VS V2")
print("-" * 100)
print(
    direction_feature_comparison_df.to_string(
        index=False
    )
)

print("\n2. SCORE CHANGE")
print("-" * 100)
print(
    f"V1 direction score       : "
    f"{best_tuned_direction_score:.6f}"
)
print(
    f"V2 direction score       : "
    f"{best_direction_v2_score:.6f}"
)
print(
    f"Absolute change          : "
    f"{v2_absolute_improvement:.6f}"
)
print(
    f"Relative change          : "
    f"{v2_relative_improvement_pct:.2f}%"
)

print("\n3. V2 CONFUSION MATRIX")
print("-" * 100)
print(direction_v2_confusion_matrix)

print("\n4. V2 PROBABILITY SUMMARY")
print("-" * 100)
print(
    pd.Series(
        direction_v2_valid_probability
    )
    .describe()
    .to_string()
)

print("\n5. NEW V2 FEATURE IMPORTANCE")
print("-" * 100)
print(
    new_v2_feature_importance[
        [
            "feature",
            "gain_importance",
            "split_importance",
        ]
    ].to_string(index=False)
)

print("\n6. TOP 20 FEATURES OVERALL")
print("-" * 100)
print(
    direction_v2_feature_importance[
        [
            "feature",
            "gain_importance",
            "split_importance",
        ]
    ]
    .head(20)
    .to_string(index=False)
)

print("\n" + "=" * 100)
print("END OF SUMMARY")
print("=" * 100)

#### Direction Feature Ablation Tests

The full V2 feature set underperformed V1.

We will now add the strongest new features to V1 in small groups and test each group independently.

Current benchmark:

- Direction V1 score: 0.294817

In [ ]:
direction_ablation_feature_sets = {
    "V1 Baseline": [],
    "V1 + Relative Return": [
        "relative_return_1d",
    ],
    "V1 + Distance from High Low": [
        "distance_from_20d_high_pct",
        "distance_from_20d_low_pct",
    ],
    "V1 + ADX DI": [
        "adx_14",
        "di_spread_14",
    ],
    "V1 + Supertrend Distance": [
        "supertrend_distance_pct_7_1",
    ],
    "V1 + Momentum Acceleration": [
        "momentum_acceleration_5d_20d",
    ],
    "V1 + Bollinger ZScore": [
        "bollinger_zscore_20d",
    ],
}

In [ ]:
def run_direction_ablation_experiment(experiment_name, added_features):
    experiment_numeric_features = (
        base_numeric_feature_columns
        + added_features
    )

    experiment_features = (
        experiment_numeric_features
        + ["symbol"]
    )

    X_train_experiment = train_v2_df[
        experiment_features
    ].copy().reset_index(drop=True)

    X_valid_experiment = valid_v2_df[
        experiment_features
    ].copy().reset_index(drop=True)

    X_train_experiment["symbol"] = (
        X_train_experiment["symbol"]
        .astype(str)
        .astype(symbol_dtype)
    )

    X_valid_experiment["symbol"] = (
        X_valid_experiment["symbol"]
        .astype(str)
        .astype(symbol_dtype)
    )

    train_dataset_experiment = lgb.Dataset(
        X_train_experiment,
        label=y_train_v2_binary,
        categorical_feature=["symbol"],
        free_raw_data=False,
    )

    valid_dataset_experiment = lgb.Dataset(
        X_valid_experiment,
        label=y_valid_v2_binary,
        categorical_feature=["symbol"],
        free_raw_data=False,
    )

    start_time = time.perf_counter()

    experiment_model = lgb.train(
        params=direction_v2_params,
        train_set=train_dataset_experiment,
        valid_sets=[valid_dataset_experiment],
        valid_names=["valid"],
        num_boost_round=1000,
        callbacks=[
            lgb.early_stopping(75, verbose=False),
            lgb.log_evaluation(0),
        ],
    )

    runtime_seconds = (
        time.perf_counter()
        - start_time
    )

    validation_probability = experiment_model.predict(
        X_valid_experiment,
        num_iteration=experiment_model.best_iteration,
    )

    threshold_results = evaluate_thresholds(
        model_name=experiment_name,
        probability_up=validation_probability,
        actual_return_pct=valid_v2_actual_return_pct,
        actual_direction=y_valid_v2,
        thresholds=threshold_grid,
    )

    best_result = threshold_results.sort_values(
        "direction_score",
        ascending=False,
    ).iloc[0]

    result = {
        "experiment": experiment_name,
        "added_features": ", ".join(added_features) if added_features else "None",
        "total_numeric_features": len(experiment_numeric_features),
        "best_iteration": int(experiment_model.best_iteration),
        "best_threshold": float(best_result["threshold"]),
        "direction_score": float(best_result["direction_score"]),
        "directional_return_pct": float(best_result["directional_return_pct"]),
        "hit_rate": float(best_result["hit_rate"]),
        "roc_auc": float(best_result["roc_auc"]),
        "predicted_up_fraction": float(best_result["predicted_up_fraction"]),
        "predicted_down_fraction": float(best_result["predicted_down_fraction"]),
        "runtime_seconds": runtime_seconds,
    }

    return {
        "result": result,
        "model": experiment_model,
        "probability": validation_probability,
        "threshold_results": threshold_results,
        "feature_columns": experiment_numeric_features,
    }

In [ ]:
direction_ablation_outputs = {}
direction_ablation_results = []

for experiment_name, added_features in direction_ablation_feature_sets.items():
    output = run_direction_ablation_experiment(
        experiment_name=experiment_name,
        added_features=added_features,
    )

    direction_ablation_outputs[experiment_name] = output
    direction_ablation_results.append(
        output["result"]
    )

    print(
        f"{experiment_name} | "
        f"Score: {output['result']['direction_score']:.6f} | "
        f"Threshold: {output['result']['best_threshold']:.2f}"
    )

In [ ]:
direction_ablation_results_df = pd.DataFrame(
    direction_ablation_results
)

direction_ablation_results_df = (
    direction_ablation_results_df
    .sort_values(
        "direction_score",
        ascending=False,
    )
    .reset_index(drop=True)
)

direction_ablation_results_df

In [ ]:
direction_ablation_results_df["score_change_vs_v1"] = (
    direction_ablation_results_df["direction_score"]
    - best_tuned_direction_score
)

direction_ablation_results_df["relative_change_vs_v1_pct"] = (
    direction_ablation_results_df["score_change_vs_v1"]
    / best_tuned_direction_score
    * 100
)

direction_ablation_results_df[
    [
        "experiment",
        "added_features",
        "best_iteration",
        "best_threshold",
        "direction_score",
        "score_change_vs_v1",
        "relative_change_vs_v1_pct",
        "hit_rate",
        "roc_auc",
        "predicted_up_fraction",
        "predicted_down_fraction",
    ]
]

In [ ]:
best_ablation_row = direction_ablation_results_df.iloc[0]

best_ablation_experiment = best_ablation_row["experiment"]
best_ablation_score = float(
    best_ablation_row["direction_score"]
)

best_ablation_output = direction_ablation_outputs[
    best_ablation_experiment
]

print("Best experiment:", best_ablation_experiment)
print("Added features:", best_ablation_row["added_features"])
print("Direction score:", best_ablation_score)
print(
    "Change vs V1:",
    best_ablation_row["score_change_vs_v1"],
)
print(
    "Relative change:",
    best_ablation_row["relative_change_vs_v1_pct"],
)

In [ ]:
# print("=" * 100)
# print("DIRECTION FEATURE ABLATION SUMMARY")
# print("=" * 100)

# print("\nCURRENT V1 BENCHMARK")
# print("-" * 100)
# print(f"Direction score: {best_tuned_direction_score:.6f}")

# print("\nABLATION RESULTS")
# print("-" * 100)

# print(
#     direction_ablation_results_df[
#         [
#             "experiment",
#             "best_iteration",
#             "best_threshold",
#             "direction_score",
#             "score_change_vs_v1",
#             "relative_change_vs_v1_pct",
#             "hit_rate",
#             "roc_auc",
#             "predicted_up_fraction",
#             "predicted_down_fraction",
#         ]
#     ].to_string(index=False)
# )

# print("\nBEST EXPERIMENT")
# print("-" * 100)
# print(f"Experiment: {best_ablation_experiment}")
# print(f"Added features: {best_ablation_row['added_features']}")
# print(f"Direction score: {best_ablation_score:.6f}")
# print(
#     f"Absolute change: "
#     f"{best_ablation_row['score_change_vs_v1']:.6f}"
# )
# print(
#     f"Relative change: "
#     f"{best_ablation_row['relative_change_vs_v1_pct']:.2f}%"
# )

# print("\n" + "=" * 100)
# print("END OF SUMMARY")
# print("=" * 100)

#### Direction Model V3

This experiment loads the precomputed V3 modelling panel. No raw daily or minute
processing is performed in this notebook.

The V3 model uses the same LightGBM parameters and validation-threshold procedure as
the V1 benchmark so that the comparison is directly interpretable.

In [ ]:
V3_MODEL_PANEL_PATH = (
    PROCESSED_DATA_DIR / "direction_v3_model_panel.parquet"
)

direction_v3_df = pd.read_parquet(
    V3_MODEL_PANEL_PATH
)

direction_v3_df["pred_date"] = pd.to_datetime(
    direction_v3_df["pred_date"]
)

# Use only columns that actually exist in model_df.
candidate_reference_columns = [
    "symbol",
    "pred_date",
    "target_date",
    "split",
    "actual_return_pct",
    "actual_direction",
    "actual_magnitude_pct",
]

v1_reference_columns = [
    column
    for column in candidate_reference_columns
    if column in model_df.columns
]

print("Reference columns used:", v1_reference_columns)

required_reference_columns = {
    "symbol",
    "pred_date",
    "split",
    "actual_direction",
}

missing_required_columns = (
    required_reference_columns
    - set(v1_reference_columns)
)

assert not missing_required_columns, (
    f"Missing required columns in model_df: "
    f"{sorted(missing_required_columns)}"
)

v1_reference_df = model_df[
    v1_reference_columns
].copy()

v1_reference_df["pred_date"] = pd.to_datetime(
    v1_reference_df["pred_date"]
)

assert not v1_reference_df.duplicated(
    ["symbol", "pred_date"]
).any()

columns_to_replace = [
    column
    for column in v1_reference_columns
    if column not in ["symbol", "pred_date"]
    and column in direction_v3_df.columns
]

direction_v3_df = direction_v3_df.drop(
    columns=columns_to_replace
)

direction_v3_df = direction_v3_df.merge(
    v1_reference_df,
    on=["symbol", "pred_date"],
    how="left",
    validate="one_to_one",
)

print("V3 panel shape:", direction_v3_df.shape)
print("Symbols:", direction_v3_df["symbol"].nunique())

print(
    direction_v3_df["split"]
    .value_counts(dropna=False)
)

In [ ]:
assert not direction_v3_df.duplicated(
    ["symbol", "pred_date"]
).any()

assert direction_v3_df["symbol"].nunique() == 208

assert direction_v3_df["split"].notna().all()

assert direction_v3_df["split"].isin(
    ["train", "valid", "test", "embargo"]
).all()

assert direction_v3_df[
    "actual_direction"
].notna().all()

print("V3 split and direction-target alignment passed.")

In [ ]:
train_v3 = direction_v3_df[
    direction_v3_df["split"] == "train"
].copy()

valid_v3 = direction_v3_df[
    direction_v3_df["split"] == "valid"
].copy()

print("Train rows:", len(train_v3))
print("Validation rows:", len(valid_v3))
print("Embargo rows excluded:", (direction_v3_df["split"] == "embargo").sum())

In [ ]:
V3_ADDED_FEATURES = [
    "universe_return_1d",
    "market_breadth",
    "cross_sectional_return_dispersion",
    "aggregate_universe_volatility",
    "idiosyncratic_return_1d",
    "relative_return_rank_1d",
    "relative_momentum_rank_5d",
    "breadth_shock_zscore",
    "dispersion_regime_zscore",
    "standardized_current_gap",
    "gap_intraday_reversal",
    "intraday_path_efficiency",
    "intraday_realized_volatility",
    "intraday_realized_skewness",
    "intraday_up_minute_fraction",
    "late_session_volatility_share_60m",
    "early_session_volatility_share_60m",
    "largest_minute_move_share",
    "signed_closing_volume_pressure",
    "closing_volume_share_60m",
    "closing_return_60m",
    "closing_trend_slope",
    "intraday_high_low_range_pct",
    "close_location_in_range",
    "volume_concentration_hhi",
    "minute_bar_coverage",
    "minute_features_available",
]

missing_v3_features = [
    column
    for column in V3_ADDED_FEATURES
    if column not in direction_v3_df.columns
]

assert not missing_v3_features, (
    f"Missing V3 features: {missing_v3_features}"
)

print("V3 added features:", len(V3_ADDED_FEATURES))

In [ ]:
print("Existing feature list variables:")

for variable_name in [
    "feature_columns",
    "FEATURE_COLUMNS",
    "direction_features",
    "model_features",
]:
    if variable_name in globals():
        print(
            variable_name,
            len(globals()[variable_name]),
        )

In [ ]:
# Show all variables containing 'feature'
for name in globals():
    if "feature" in name.lower():
        print(name)

In [ ]:
# Recover V1 features that were not carried into the saved V3 panel.
missing_v1_features = [
    column
    for column in lightgbm_feature_columns
    if column not in direction_v3_df.columns
]

print("V1 features to recover:", missing_v1_features)

v1_feature_reference = model_df[
    [
        "symbol",
        "pred_date",
        *missing_v1_features,
    ]
].copy()

v1_feature_reference["pred_date"] = pd.to_datetime(
    v1_feature_reference["pred_date"]
)

assert not v1_feature_reference.duplicated(
    ["symbol", "pred_date"]
).any()

direction_v3_df = direction_v3_df.merge(
    v1_feature_reference,
    on=["symbol", "pred_date"],
    how="left",
    validate="one_to_one",
)

print("V3 shape after recovering V1 features:", direction_v3_df.shape)

print(
    direction_v3_df[
        missing_v1_features
    ].isna().sum()
)

In [ ]:
V3_FEATURE_COLUMNS = list(
    dict.fromkeys(
        lightgbm_feature_columns
        + V3_ADDED_FEATURES
    )
)

missing_model_features = [
    column
    for column in V3_FEATURE_COLUMNS
    if column not in direction_v3_df.columns
]

assert not missing_model_features, (
    f"Missing model features: {missing_model_features}"
)

print("V1 feature count:", len(lightgbm_feature_columns))
print("V3 added feature count:", len(V3_ADDED_FEATURES))
print("Final V3 feature count:", len(V3_FEATURE_COLUMNS))

In [ ]:
train_v3 = direction_v3_df[
    direction_v3_df["split"] == "train"
].copy()

valid_v3 = direction_v3_df[
    direction_v3_df["split"] == "valid"
].copy()

print("Train rows:", len(train_v3))
print("Validation rows:", len(valid_v3))

In [ ]:
X_train_v3 = train_v3[
    V3_FEATURE_COLUMNS
].copy()

X_valid_v3 = valid_v3[
    V3_FEATURE_COLUMNS
].copy()

y_train_v3 = (
    train_v3["actual_direction"]
    .eq(1)
    .astype("int8")
)

y_valid_v3 = (
    valid_v3["actual_direction"]
    .eq(1)
    .astype("int8")
)

for frame in [
    X_train_v3,
    X_valid_v3,
]:
    frame["symbol"] = frame[
        "symbol"
    ].astype("category")

print("X_train_v3:", X_train_v3.shape)
print("X_valid_v3:", X_valid_v3.shape)
print("Train positive share:", y_train_v3.mean())
print("Valid positive share:", y_valid_v3.mean())

In [ ]:
print(
    X_train_v3.dtypes
    .value_counts()
)

numeric_v3_features = [
    column
    for column in V3_FEATURE_COLUMNS
    if column != "symbol"
]

assert not np.isinf(
    X_train_v3[
        numeric_v3_features
    ].to_numpy(dtype=float)
).any()

assert not np.isinf(
    X_valid_v3[
        numeric_v3_features
    ].to_numpy(dtype=float)
).any()

print("Feature matrix checks passed.")

In [ ]:
import lightgbm as lgb

v3_direction_model = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=1000,
    num_leaves=31,
    learning_rate=0.05,
    min_child_samples=100,
    feature_fraction=0.60,
    bagging_fraction=0.80,
    bagging_freq=1,
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
)

v3_direction_model.fit(
    X_train_v3,
    y_train_v3,
    categorical_feature=["symbol"],
    eval_set=[
        (X_valid_v3, y_valid_v3),
    ],
    eval_metric="binary_logloss",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=75,
            verbose=True,
        )
    ],
)

print(
    "Best iteration:",
    v3_direction_model.best_iteration_,
)

In [ ]:
valid_v3_probability_up = (
    v3_direction_model.predict_proba(
        X_valid_v3,
        num_iteration=v3_direction_model.best_iteration_,
    )[:, 1]
)

pd.Series(
    valid_v3_probability_up
).describe()

In [ ]:
threshold_results_v3 = []

actual_returns_v3 = valid_v3[
    "actual_return_pct"
].to_numpy()

actual_directions_v3 = valid_v3[
    "actual_direction"
].to_numpy()

for threshold in np.arange(
    0.40,
    0.801,
    0.01,
):
    predicted_direction_v3 = np.where(
        valid_v3_probability_up >= threshold,
        1,
        -1,
    )

    direction_score_v3 = (
        np.sum(
            predicted_direction_v3
            * actual_returns_v3
        )
        / np.sum(
            np.abs(actual_returns_v3)
        )
    )

    hit_rate_v3 = np.mean(
        predicted_direction_v3
        == actual_directions_v3
    )

    threshold_results_v3.append(
        {
            "threshold": threshold,
            "direction_score": direction_score_v3,
            "hit_rate": hit_rate_v3,
            "predicted_up_fraction": np.mean(
                predicted_direction_v3 == 1
            ),
        }
    )

threshold_results_v3 = (
    pd.DataFrame(
        threshold_results_v3
    )
    .sort_values(
        "direction_score",
        ascending=False,
    )
    .reset_index(drop=True)
)

threshold_results_v3.head(10)

In [ ]:
reconciled_direction_feature_columns = lightgbm_feature_columns.copy()

reconciled_train_df = train_df.copy()
reconciled_valid_df = valid_df.copy()

reconciled_symbol_categories = sorted(
    direction_df["symbol"].astype(str).unique()
)

reconciled_symbol_dtype = pd.CategoricalDtype(
    categories=reconciled_symbol_categories
)

X_train_reconciled = reconciled_train_df[
    reconciled_direction_feature_columns
].copy()

X_valid_reconciled = reconciled_valid_df[
    reconciled_direction_feature_columns
].copy()

X_train_reconciled["symbol"] = X_train_reconciled["symbol"].astype(str).astype(
    reconciled_symbol_dtype
)
X_valid_reconciled["symbol"] = X_valid_reconciled["symbol"].astype(str).astype(
    reconciled_symbol_dtype
)

reconciled_direction_model = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=1000,
    num_leaves=31,
    learning_rate=0.05,
    min_child_samples=100,
    feature_fraction=0.60,
    bagging_fraction=0.80,
    bagging_freq=1,
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
)

reconciled_direction_model.fit(
    X_train_reconciled,
    reconciled_train_df["actual_direction"].eq(1).astype("int8"),
    categorical_feature=["symbol"],
)

reconciled_valid_probability_up = reconciled_direction_model.predict_proba(
    X_valid_reconciled
)[:, 1]

reconciled_valid_prediction = np.where(
    reconciled_valid_probability_up >= 0.64,
    1,
    -1,
)

reconciled_v1_validation_metrics = evaluate_direction_model(
    model_name="Reconciled V1 Final Validation Path",
    actual_return_pct=valid_actual_return_pct,
    actual_direction=y_valid,
    predicted_direction=reconciled_valid_prediction,
    probability_up=reconciled_valid_probability_up,
    runtime_seconds=0.0,
)

reconciled_v1_validation_metrics["threshold"] = 0.64
reconciled_v1_validation_metrics

In [ ]:
best_v3_result = (
    threshold_results_v3.iloc[0]
)

V1_BENCHMARK_SCORE = reconciled_v1_validation_metrics["direction_score"]

print(
    "Best V3 threshold:",
    round(
        best_v3_result["threshold"],
        2,
    ),
)

print(
    "Best V3 direction score:",
    round(
        best_v3_result[
            "direction_score"
        ],
        6,
    ),
)

print(
    "V3 hit rate:",
    round(
        best_v3_result[
            "hit_rate"
        ],
        6,
    ),
)

print(
    "V3 predicted-up fraction:",
    round(
        best_v3_result[
            "predicted_up_fraction"
        ],
        6,
    ),
)

print(
    "V1 benchmark:",
    V1_BENCHMARK_SCORE,
)

print(
    "Difference:",
    round(
        best_v3_result[
            "direction_score"
        ]
        - V1_BENCHMARK_SCORE,
        6,
    ),
)

In [ ]:
direction_model_decision = {
    "selected_version": "V1",
    "selected_threshold": 0.64,
    "selected_validation_direction_score": float(reconciled_v1_validation_metrics["direction_score"]),
    "rejected_version": "V3",
    "rejected_validation_direction_score": float(
        best_v3_result["direction_score"]
    ),
    "reason": (
        "V3 underperformed the directly comparable V1 benchmark. "
        "Further direction feature engineering was stopped to prioritise "
        "magnitude and confidence modelling."
    ),
}

direction_model_decision

In [ ]:
direction_v3_experiment_path = (
    PROCESSED_DATA_DIR / "direction_v3_experiment_summary.csv"
)

pd.DataFrame(
    [
        {
            "version": "V1",
            "threshold": 0.64,
            "direction_score": float(reconciled_v1_validation_metrics["direction_score"]),
            "selected": True,
        },
        {
            "version": "V3",
            "threshold": float(
                best_v3_result["threshold"]
            ),
            "direction_score": float(
                best_v3_result["direction_score"]
            ),
            "selected": False,
        },
    ]
).to_csv(
    direction_v3_experiment_path,
    index=False,
)

print("Saved:", direction_v3_experiment_path)

#### Final Direction Model Decision

The V1 pooled LightGBM classifier is retained as the final direction model.

- Validation threshold: 0.64
- Validation direction score: 0.294817
- V3 validation direction score: 0.311213

V3 added cross-sectional, regime, gap-behaviour, and minute-derived features but
materially underperformed the directly comparable V1 benchmark. Further direction
experimentation was stopped to prioritise magnitude and confidence modelling.

The test split has not been used for model or threshold selection.

> During production reconciliation, a stale direction-score reference of 0.371872 was identified. A row-for-row comparison between the cached notebook panel and the raw-data rebuild produced identical features and predictions. The reproducible validation direction score is 0.294817, and all final artifacts use that value.

In [ ]:
FINAL_DIRECTION_VERSION = "V1"
FINAL_DIRECTION_THRESHOLD = 0.64
FINAL_DIRECTION_VALID_SCORE = float(reconciled_v1_validation_metrics["direction_score"])

direction_model_decision = {
    "selected_version": FINAL_DIRECTION_VERSION,
    "model_family": "LightGBM binary classifier",
    "model_structure": "pooled model with symbol as categorical feature",
    "selected_threshold": FINAL_DIRECTION_THRESHOLD,
    "validation_direction_score": FINAL_DIRECTION_VALID_SCORE,
    "v3_threshold": float(best_v3_result["threshold"]),
    "v3_validation_direction_score": float(
        best_v3_result["direction_score"]
    ),
    "test_used_for_selection": False,
    "decision": (
        "Retain V1 because V3 materially underperformed on validation. "
        "Stop direction iteration and move to magnitude modelling."
    ),
}

direction_model_decision

In [ ]:
direction_version_comparison = pd.DataFrame(
    [
        {
            "version": "V1",
            "feature_set": "original",
            "threshold": reconciled_v1_validation_metrics["threshold"],
            "validation_direction_score": float(reconciled_v1_validation_metrics["direction_score"]),
            "validation_hit_rate": float(reconciled_v1_validation_metrics["hit_rate"]),
            "selected": True,
            "status": "recomputed_executable",
            "note": "Validated row-for-row against raw-data production rebuild.",
        },
        {
            "version": "V2",
            "feature_set": "technical indicator additions",
            "threshold": np.nan,
            "validation_direction_score": 0.366080,
            "validation_hit_rate": np.nan,
            "selected": False,
            "status": "historical_noncomparable",
            "note": "Not revalidated during production reconciliation; historical notebook record only.",
        },
        {
            "version": "V3",
            "feature_set": (
                "cross-sectional, regime, gap and minute-derived"
            ),
            "threshold": float(best_v3_result["threshold"]),
            "validation_direction_score": float(
                best_v3_result["direction_score"]
            ),
            "validation_hit_rate": float(
                best_v3_result["hit_rate"]
            ),
            "selected": False,
            "status": "recomputed_executable",
            "note": "Directly comparable executable rerun on the same validation rows.",
        },
    ]
)

direction_version_comparison

In [ ]:
DIRECTION_RESULTS_PATH = (
    PROCESSED_DATA_DIR
    / "direction_model_version_comparison.csv"
)

direction_version_comparison.to_csv(
    DIRECTION_RESULTS_PATH,
    index=False,
)

print("Saved:", DIRECTION_RESULTS_PATH)

In [ ]:
final_direction_config = {
    "selected_version": "V1",
    "model_family": "LightGBM",
    "objective": "binary",
    "threshold": 0.64,
    "validation_direction_score": FINAL_DIRECTION_VALID_SCORE,
    "validation_hit_rate": float(reconciled_v1_validation_metrics["hit_rate"]),
    "validation_score_source": "executable_validation_stage_v1_reconciliation",
    "hyperparameters": {
        "num_leaves": 31,
        "learning_rate": 0.05,
        "min_data_in_leaf": 100,
        "feature_fraction": 0.60,
        "bagging_fraction": 0.80,
        "random_state": 42,
    },
    "categorical_features": ["symbol"],
    "selection_split": "valid",
    "test_used_for_selection": False,
}

with open(
    FINAL_DIRECTION_CONFIG_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        final_direction_config,
        file,
        indent=2,
    )

print("Saved:", FINAL_DIRECTION_CONFIG_PATH)

In [ ]:
assert FINAL_DIRECTION_VERSION == "V1"
assert FINAL_DIRECTION_THRESHOLD == 0.64
assert np.isclose(FINAL_DIRECTION_VALID_SCORE, 0.29481723269301396)

assert DIRECTION_RESULTS_PATH.exists()
assert FINAL_DIRECTION_CONFIG_PATH.exists()

print("Direction notebook complete.")
print("Selected model: V1 LightGBM")
print("Validation threshold: 0.64")
print(f"Validation direction score: {FINAL_DIRECTION_VALID_SCORE:.6f}")
print("Next notebook: 04_magnitude_model.ipynb")

In [ ]:
MAGNITUDE_PANEL_PATH = (
    PROCESSED_DATA_DIR / "magnitude_model_panel.parquet"
)

model_df.to_parquet(
    MAGNITUDE_PANEL_PATH,
    index=False,
)

print("Saved:", MAGNITUDE_PANEL_PATH)
print(model_df.shape)

In [ ]:
import json

DIRECTION_FEATURE_LIST_PATH = (
    PROCESSED_DATA_DIR / "final_direction_feature_columns.json"
)

with open(
    DIRECTION_FEATURE_LIST_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        lightgbm_feature_columns,
        file,
        indent=2,
    )

print("Saved:", DIRECTION_FEATURE_LIST_PATH.resolve())
print("Feature count:", len(lightgbm_feature_columns))